# Notebook 07 — Visual Odometry (Classical + Learned)

**Vision & 3D Mapping Workshop** | Block 3: Multi-View Geometry

---

## Why This Matters

Visual Odometry (VO) estimates a camera's motion from a sequence of images.  It is the
backbone of autonomous driving, drone navigation, augmented reality, and mobile robotics.

This notebook takes you from the mathematical foundations — epipolar geometry, the essential
matrix, triangulation — all the way to a working monocular VO pipeline and a discussion of
modern learned approaches.  Every key equation is **derived**, not just stated.

### What You'll Learn

1. **Epipolar Geometry** — full derivation of the essential matrix
2. **8-Point Algorithm** — recovering $E$ from correspondences (+ 5-point algorithm)
3. **Essential Matrix Decomposition** — extracting $(R, \mathbf{t})$ from $E$
4. **Triangulation** — DLT, midpoint, Hartley-Sturm, Lindström
5. **Monocular VO Pipeline** — putting it all together
6. **Scale Ambiguity** — the fundamental limitation of monocular VO
7. **PnP for Stereo VO** — solving VO with known 3D structure
8. **Learned VO** — DPVO, DPV-SLAM, DROID-SLAM, FoundationSLAM
9. **Nonlinear Least Squares** — Gauss-Newton and Levenberg-Marquardt for pose refinement
10. **Exercises** — hands-on practice

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial.transform import Rotation
from scipy.optimize import least_squares

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

---

## Shared Utilities

The following helper functions are used throughout this notebook: camera projection,
skew-symmetric matrix construction, 3D point generation, rotation via Rodrigues'
formula, and a `look_at` function for positioning virtual cameras.

In [ ]:
def skew(v):
    """Skew-symmetric matrix from 3-vector."""
    v = v.flatten()
    return np.array([
        [0,    -v[2],  v[1]],
        [v[2],  0,    -v[0]],
        [-v[1], v[0],  0   ]
    ])


def make_K(fx=500.0, fy=500.0, cx=320.0, cy=240.0):
    """Camera intrinsic matrix."""
    return np.array([[fx, 0, cx],
                     [0, fy, cy],
                     [0,  0,  1]], dtype=np.float64)


def project(points_3d, K, R, t):
    """Project 3D world points to 2D pixel coordinates.
    points_3d: (N, 3), R: (3, 3) world-to-cam rotation, t: (3,) world-to-cam translation.
    Returns: (N, 2) pixel coordinates.
    """
    t = t.reshape(3, 1)
    P = K @ np.hstack([R, t])                             # (3, 4)
    pts_h = np.hstack([points_3d, np.ones((len(points_3d), 1))]).T  # (4, N)
    proj = P @ pts_h                                      # (3, N)
    proj = proj[:2] / proj[2:3]                           # (2, N)
    return proj.T                                         # (N, 2)

In [ ]:
def generate_3d_points(n=200, xlim=(-5, 5), ylim=(-5, 5), zlim=(5, 15)):
    """Random 3D point cloud in a box."""
    return np.random.uniform(
        low=[xlim[0], ylim[0], zlim[0]],
        high=[xlim[1], ylim[1], zlim[1]],
        size=(n, 3)
    )


def rotation_matrix(axis, angle):
    """Rotation matrix via Rodrigues' formula."""
    axis = np.asarray(axis, dtype=np.float64)
    n = np.linalg.norm(axis)
    if n < 1e-15:
        return np.eye(3)
    axis = axis / n
    K = skew(axis)
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


def look_at(cam_pos, target, up=np.array([0.0, -1.0, 0.0])):
    """Compute world-to-camera (R, t) so the camera at cam_pos looks at target."""
    z_axis = target - cam_pos
    z_axis = z_axis / np.linalg.norm(z_axis)
    x_axis = np.cross(up, z_axis)
    x_norm = np.linalg.norm(x_axis)
    if x_norm < 1e-8:
        up = np.array([1.0, 0.0, 0.0])
        x_axis = np.cross(up, z_axis)
    x_axis = x_axis / np.linalg.norm(x_axis)
    y_axis = np.cross(z_axis, x_axis)
    R_cw = np.stack([x_axis, y_axis, z_axis], axis=0)
    t_cw = -R_cw @ cam_pos
    return R_cw, t_cw

In [ ]:
K_test = make_K()
print("Camera intrinsic matrix K:")
print(K_test)

v = np.array([1.0, 2.0, 3.0])
S = skew(v)
print(f"\nskew({v}) = ")
print(S)
print(f"S + S^T (should be 0):\n{S + S.T}")

## The Visual Odometry Problem — Mathematical Formulation

### Sequential Pose Estimation

Visual Odometry estimates the camera's trajectory by computing the relative motion
between consecutive frames. The camera pose at time $k$ is a rigid-body
transformation $T_k \in \text{SE}(3)$, recovered by chaining incremental motions:

$$T_k = T_{k-1} \cdot \Delta T_{k-1,k}$$

The full trajectory from the initial frame is the composition:

$$T_k = T_0 \cdot \prod_{i=1}^{k} \Delta T_{i-1,i}$$

This **sequential composition** is both the strength and the weakness of VO: it
enables real-time incremental computation, but errors in each $\Delta T$ **accumulate**
over time (drift).

### The Motion Estimation Pipeline

Every classical VO system follows a four-stage pipeline:

| Stage | Input | Output | Key Algorithms |
|-------|-------|--------|----------------|
| **Detect** | Image $I_k$ | Keypoints $\{\mathbf{p}_i\}$ | ORB, SIFT, SuperPoint |
| **Match** | Keypoints in $I_{k-1}, I_k$ | Correspondences $\{(\mathbf{p}_i, \mathbf{p}'_i)\}$ | BF, FLANN, LightGlue |
| **Estimate** | Correspondences + $K$ | Relative pose $\Delta T_{k-1,k}$ | 5-point + RANSAC, PnP |
| **Refine** | Initial $\Delta T$ + all inliers | Optimized $\Delta T^*$ | Gauss-Newton, LM, local BA |

### Scale Ambiguity in Monocular VO

A fundamental limitation: the essential matrix $E = [\mathbf{t}]_\times R$ encodes
translation **direction** only. For any $\alpha > 0$:

$$\alpha E = [\alpha\mathbf{t}]_\times R$$

produces identical epipolar geometry. Monocular VO therefore recovers the trajectory in
the **similarity group** $\text{Sim}(3)$, not $\text{SE}(3)$. The trajectory *shape* is
correct, but absolute distances are unknown without an external scale source (stereo
baseline, IMU, known object sizes).

### VO vs. SLAM

|  | **Visual Odometry** | **SLAM** |
|---|---|---|
| **Scope** | Local motion (frame-to-frame) | Global map + trajectory |
| **Map** | Transient — discarded after use | Persistent, globally consistent |
| **Loop closure** | Not handled | Detects revisited places → corrects drift |
| **Drift** | Accumulates unboundedly | Bounded by loop closures |
| **Complexity** | $O(W)$ per frame (sliding window of size $W$) | $O(n)$ to $O(n^2)$ without sparsity tricks |

VO is the **front-end** of most SLAM systems: it provides local motion estimates that
SLAM's **back-end** (pose-graph optimization, bundle adjustment) refines into a globally
consistent trajectory.

> **Reference:** Scaramuzza, D. & Fraundorfer, F., "Visual Odometry — Part I: The
> First 30 Years and Fundamentals," *IEEE Robotics & Automation Magazine*, 2011;
> "Part II: Matching, Robustness, Optimization, and Applications," 2012.

## Feature-Based vs. Direct Methods — The Two VO Paradigms

Before diving into the mathematical machinery of epipolar geometry, it is important to
understand the two fundamentally different philosophies that have shaped Visual Odometry.

### Feature-Based (Indirect) Methods

Detect sparse keypoints → compute descriptors → match across frames → estimate geometry
from correspondences → minimize **geometric (reprojection) error**.

- **Pros:** Robust to illumination changes; computationally efficient (few hundred
  keypoints per frame); wide convergence basin handles large inter-frame motion
- **Cons:** Discards ~99% of pixel information; fails in textureless regions (no corners);
  detection thresholds need tuning per environment
- **Key systems:** ORB-SLAM3 (Campos et al., TRO 2021), VINS-Fusion (Qin et al., TRO 2018)

### Direct (Photometric) Methods

Skip feature extraction entirely. Minimize the **photometric error** between warped
pixel intensities:

$$\mathcal{L}_{\text{photo}} = \sum_{\mathbf{p} \in \Omega} \rho\!\left( I_1(\mathbf{p}) - I_2\bigl(\pi(T_{12} \cdot \pi^{-1}(\mathbf{p},\, d(\mathbf{p})))\bigr) \right)$$

- **Pros:** Uses all image information; works in textureless regions; produces denser maps
- **Cons:** Sensitive to auto-exposure / motion blur (brightness constancy assumption);
  small convergence basin requires good initialization; needs photometric calibration
- **Key systems:** DSO (Engel et al., TPAMI 2018), LSD-SLAM (Engel et al., ECCV 2014)

### Semi-Direct: The Best of Both Worlds

**SVO** (Forster et al., TRO 2017) tracks sparse features via photometric patch
alignment (fast, sub-pixel accurate) and uses feature correspondences for joint
optimization — achieving the speed of direct methods with the initialization robustness
of feature-based approaches.

### The 2025–2026 Paradigm Shift: Feed-Forward Foundation Models

The classical detect → match → estimate pipeline is being challenged by **feed-forward
foundation models** that bypass it entirely:

| System | Year | Key Innovation |
|--------|------|----------------|
| **DUSt3R** | CVPR 2024 | Two images → dense 3D point maps; no features, no matching, no RANSAC |
| **MASt3R** | 2024 | Adds dense local features to DUSt3R for retrieval + multi-view extension |
| **VGGT** | 2025 | Single Transformer predicts cameras, points, depth, and correspondences jointly |

These models are trained on massive 3D datasets and **generalize** across domains without
fine-tuning, representing a shift from "handcrafted pipeline" to "learned geometric prior."

> **Reference:** Luo, Y. et al., "Why does Deep Learning Improve Visual SLAM?",
> *arXiv 2607.06023*, July 2026.

---

## 1. Epipolar Geometry — Full Derivation

### Setup

Consider a 3D point $\mathbf{P}$ observed by two cameras.  Camera 1 sits at the world
origin; camera 2 is related by rotation $R$ and translation $\mathbf{t}$ (the world-to-camera
transform for camera 2):

$$\mathbf{P}_1 = \mathbf{P}, \qquad \mathbf{P}_2 = R\,\mathbf{P} + \mathbf{t}$$

The point projects to **normalized image coordinates** (dividing by the depth $Z$):

$$\mathbf{p}_1 = \frac{1}{Z_1}\mathbf{P}_1, \qquad \mathbf{p}_2 = \frac{1}{Z_2}\mathbf{P}_2$$

### The Coplanarity Constraint

The key geometric insight is that the vectors $\mathbf{p}_1$, $\mathbf{p}_2$, and the
baseline $\mathbf{t}$ (connecting the two camera centers) all lie in a single plane — the
**epipolar plane**.

Starting from $\mathbf{P}_2 = R\,\mathbf{P}_1 + \mathbf{t}$, substitute the depth-scaled
normalized coordinates:

$$Z_2\,\mathbf{p}_2 = R\,(Z_1\,\mathbf{p}_1) + \mathbf{t}$$

Take the cross product with $\mathbf{t}$ on both sides:

$$Z_2\,(\mathbf{t} \times \mathbf{p}_2) = Z_1\,(\mathbf{t} \times R\,\mathbf{p}_1) + \underbrace{\mathbf{t} \times \mathbf{t}}_{=\,\mathbf{0}}$$

Now dot both sides with $\mathbf{p}_2$.  The left side vanishes because the cross product
$\mathbf{t} \times \mathbf{p}_2$ is perpendicular to $\mathbf{p}_2$:

$$0 = Z_1\,\mathbf{p}_2 \cdot (\mathbf{t} \times R\,\mathbf{p}_1)$$

Since $Z_1 > 0$ (the point is in front of the camera):

$$\boxed{\mathbf{p}_2^\top\, (\mathbf{t} \times R\,\mathbf{p}_1) = 0}$$

This is the **epipolar constraint** (also called the *coplanarity constraint*).

### From Cross Product to Matrix Form

The cross product $\mathbf{t} \times \mathbf{a}$ can be written as a matrix-vector product
using the **skew-symmetric matrix** $[\mathbf{t}]_\times$:

$$\mathbf{t} \times \mathbf{a} = [\mathbf{t}]_\times\, \mathbf{a}$$

where

$$[\mathbf{t}]_\times = \begin{bmatrix} 0 & -t_z & t_y \\ t_z & 0 & -t_x \\ -t_y & t_x & 0 \end{bmatrix}$$

Substituting:

$$\mathbf{p}_2^\top\, [\mathbf{t}]_\times\, R\, \mathbf{p}_1 = 0$$

### Defining the Essential Matrix

> **Cross-reference:** The full derivation of the Fundamental and Essential matrices from first principles
> (8-point algorithm, Hartley normalization, rank-2 enforcement, epipolar line visualization)
> is in **Notebook 05 (Feature Detection & Matching), §7–8**. Here we summarize the key result
> and focus on its application in the VO pipeline.

We define the **Essential Matrix**:

$$\boxed{E = [\mathbf{t}]_\times R}$$

so the epipolar constraint becomes:

$$\mathbf{p}_2^\top\, E\, \mathbf{p}_1 = 0$$

This relates corresponding points in normalized image coordinates across two
views, encoding the relative rotation and translation between the cameras.

In **pixel** coordinates $\mathbf{x} = K\,\mathbf{p}$, the constraint becomes:

$$\mathbf{x}_2^\top\, K^{-\top} E\, K^{-1}\, \mathbf{x}_1 = 0 \;\equiv\; \mathbf{x}_2^\top\, F\, \mathbf{x}_1 = 0$$

where $F = K^{-\top} E\, K^{-1}$ is the **Fundamental Matrix** (used when $K$ is unknown).

### Properties of $E$

**Rank 2:**  The skew-symmetric matrix $[\mathbf{t}]_\times$ has rank 2 because its null
space is $\mathbf{t}$ itself ($\mathbf{t} \times \mathbf{t} = \mathbf{0}$).  Since $R$ is
full rank (orthogonal), the product $E = [\mathbf{t}]_\times R$ has rank 2.

**Two equal singular values:**  Consider $E^\top E$:

$$E^\top E = ([ \mathbf{t}]_\times R)^\top ([\mathbf{t}]_\times R) = R^\top [\mathbf{t}]_\times^\top [\mathbf{t}]_\times R$$

Since $[\mathbf{t}]_\times$ is skew-symmetric, $[\mathbf{t}]_\times^\top = -[\mathbf{t}]_\times$, so:

$$[\mathbf{t}]_\times^\top [\mathbf{t}]_\times = -[\mathbf{t}]_\times^2$$

Using the identity $[\mathbf{t}]_\times^2 = \mathbf{t}\mathbf{t}^\top - \|\mathbf{t}\|^2 I$:

$$[\mathbf{t}]_\times^\top [\mathbf{t}]_\times = \|\mathbf{t}\|^2 I - \mathbf{t}\mathbf{t}^\top$$

This matrix has eigenvalues:
- $\mathbf{t}$ is an eigenvector: eigenvalue $= \|\mathbf{t}\|^2 - \|\mathbf{t}\|^2 = 0$
- Any $\mathbf{v} \perp \mathbf{t}$: eigenvalue $= \|\mathbf{t}\|^2$

Since $R$ is orthogonal, it doesn't change eigenvalues, so $E^\top E$ has eigenvalues
$\{\|\mathbf{t}\|^2, \|\mathbf{t}\|^2, 0\}$, meaning $E$ has singular values
$\{\|\mathbf{t}\|, \|\mathbf{t}\|, 0\}$.

In [ ]:
# --- Construct the ground-truth Essential Matrix ---
np.random.seed(42)

K = make_K()

R1, t1 = np.eye(3), np.zeros(3)   # camera 1 at origin
angle = np.deg2rad(10)
R_rel = rotation_matrix([0, 1, 0], angle)
t_rel = np.array([0.5, 0.0, 0.1])

E_gt = skew(t_rel) @ R_rel
print("Ground truth Essential Matrix E:")
print(E_gt)
print(f"\nRank of E: {np.linalg.matrix_rank(E_gt, tol=1e-10)}")

In [ ]:
U, S, Vt = np.linalg.svd(E_gt)
print(f"Singular values of E: {S}")
print(f"Ratio σ₁/σ₂ = {S[0]/S[1]:.6f}  (should be 1.0)")
print(f"σ₃ = {S[2]:.2e}  (should be ≈ 0)")
print(f"||t|| = {np.linalg.norm(t_rel):.6f}, σ₁ = {S[0]:.6f}")

In [ ]:
tx = skew(t_rel)
lhs = tx.T @ tx
rhs = np.linalg.norm(t_rel)**2 * np.eye(3) - np.outer(t_rel, t_rel)

print("[t]×ᵀ [t]×:")
print(lhs)
print("\n||t||² I − t tᵀ:")
print(rhs)
print(f"\nDifference (should be ≈ 0): {np.linalg.norm(lhs - rhs):.2e}")

In [ ]:
pts_3d = generate_3d_points(n=100)

pts1_px = project(pts_3d, K, R1, t1)
pts2_px = project(pts_3d, K, R_rel, t_rel)

K_inv = np.linalg.inv(K)
pts1_norm = (K_inv @ np.hstack([pts1_px, np.ones((100, 1))]).T).T  # (N, 3)
pts2_norm = (K_inv @ np.hstack([pts2_px, np.ones((100, 1))]).T).T

errors = np.array([p2 @ E_gt @ p1 for p1, p2 in zip(pts1_norm, pts2_norm)])
print(f"Epipolar constraint p₂ᵀ E p₁ (should all be ≈ 0):")
print(f"  Max |error|:  {np.max(np.abs(errors)):.2e}")
print(f"  Mean |error|: {np.mean(np.abs(errors)):.2e}")

In [ ]:
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(pts_3d[:, 0], pts_3d[:, 2], pts_3d[:, 1],
           c='steelblue', s=10, alpha=0.5, label='3D Points')

cam1_pos = np.zeros(3)
cam2_pos = -R_rel.T @ t_rel
ax.scatter(*cam1_pos[[0, 2, 1]], c='red', s=100, marker='^', label='Camera 1')
ax.scatter(*cam2_pos[[0, 2, 1]], c='green', s=100, marker='^', label='Camera 2')

ax.plot([cam1_pos[0], cam2_pos[0]],
        [cam1_pos[2], cam2_pos[2]],
        [cam1_pos[1], cam2_pos[1]],
        'k--', linewidth=2, label='Baseline')

for i in range(10):
    P = pts_3d[i]
    ax.plot([cam1_pos[0], P[0]], [cam1_pos[2], P[2]], [cam1_pos[1], P[1]],
            'r-', alpha=0.3, linewidth=0.8)
    ax.plot([cam2_pos[0], P[0]], [cam2_pos[2], P[2]], [cam2_pos[1], P[1]],
            'g-', alpha=0.3, linewidth=0.8)

ax.set_xlabel('X'); ax.set_ylabel('Z'); ax.set_zlabel('Y')
ax.set_title('Epipolar Geometry: Two Views of a 3D Scene')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
F_gt = np.linalg.inv(K).T @ E_gt @ np.linalg.inv(K)

n_vis = 20
displacements = np.linalg.norm(pts2_px[:n_vis] - pts1_px[:n_vis], axis=1)
disp_norm = displacements / (displacements.max() + 1e-12)

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

# ── Panel 1: Feature flow vectors (colored by track displacement) ──
ax_flow = axes[0]
ax_flow.set_facecolor('#1a1a2e')
for idx in range(n_vis):
    frac = disp_norm[idx]
    color = plt.cm.plasma(frac)
    ax_flow.plot(pts1_px[idx, 0], pts1_px[idx, 1], 'o', color=color,
                 markersize=6, markeredgecolor='white', markeredgewidth=0.3)
    ax_flow.annotate('', xy=(pts2_px[idx, 0], pts2_px[idx, 1]),
                     xytext=(pts1_px[idx, 0], pts1_px[idx, 1]),
                     arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=0.8))
sm_flow = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, displacements.max()))
sm_flow.set_array([])
plt.colorbar(sm_flow, ax=ax_flow, shrink=0.8, label='Displacement (px)')
ax_flow.set_xlim(0, 640); ax_flow.set_ylim(480, 0)
ax_flow.set_xlabel('u (px)'); ax_flow.set_ylabel('v (px)')
ax_flow.set_title('Feature Flow Vectors (colored by displacement)')
ax_flow.set_aspect('equal')

# ── Panel 2: Image 1 with epipolar lines from Image 2 ──
for idx in range(15):
    color = plt.cm.tab10(idx % 10)
    x1 = np.array([pts1_px[idx, 0], pts1_px[idx, 1], 1.0])
    x2 = np.array([pts2_px[idx, 0], pts2_px[idx, 1], 1.0])

    axes[1].plot(pts1_px[idx, 0], pts1_px[idx, 1], 'o', color=color,
                 markersize=6, markeredgecolor='white', markeredgewidth=0.5)
    axes[2].plot(pts2_px[idx, 0], pts2_px[idx, 1], 'o', color=color,
                 markersize=6, markeredgecolor='white', markeredgewidth=0.5)

    l2 = F_gt @ x1
    x_range = np.array([0, 640])
    y_vals = -(l2[0] * x_range + l2[2]) / (l2[1] + 1e-12)
    axes[2].plot(x_range, y_vals, '-', color=color, alpha=0.4, linewidth=0.8)

    l1 = F_gt.T @ x2
    y_vals1 = -(l1[0] * x_range + l1[2]) / (l1[1] + 1e-12)
    axes[1].plot(x_range, y_vals1, '-', color=color, alpha=0.4, linewidth=0.8)

for ax, title in zip(axes[1:], ['Image 1 — epipolar lines from Image 2',
                                 'Image 2 — epipolar lines from Image 1']):
    ax.set_xlim(0, 640); ax.set_ylim(480, 0)
    ax.set_xlabel('u (px)'); ax.set_ylabel('v (px)')
    ax.set_title(title)
    ax.set_aspect('equal')
    ax.set_facecolor('#f5f5f5')

plt.suptitle('Feature Correspondences & Epipolar Geometry', fontsize=14)
plt.tight_layout()
plt.show()

### Understanding Feature Correspondences and Epipolar Lines

The visualization above reveals the elegant geometry that connects two views of the
same scene:

**Flow vectors (left panel):** Each arrow connects a feature's position in Image 1
to its position in Image 2, with color encoding the magnitude of displacement.
Warmer (brighter) arrows indicate features with larger apparent motion. In this
setup — a small rotation about the $Y$-axis plus a lateral translation — the flow
field shows a characteristic pattern:
- Features near the **epipole** (the vanishing point of the camera's motion direction)
  have small displacements
- Features far from the epipole have large displacements
- The flow vectors are approximately radial from the epipole — this is the
  **focus of expansion** pattern, a hallmark of forward/lateral camera motion

**Epipolar lines (center and right panels):** For each point in one image, the
corresponding point in the other image must lie on a specific line — the
**epipolar line**. This constraint reduces the correspondence search from 2D to
1D, cutting computation by a factor proportional to the image height.

Key properties visible in the plot:
- All epipolar lines in a single image pass through (or converge toward) the
  **epipole** $\mathbf{e} = K\mathbf{t}$ — the projection of one camera center
  into the other camera's image
- Corresponding points (same color) lie exactly on their epipolar lines (zero
  epipolar error), confirming that our ground-truth geometry is consistent
- The line spacing encodes the **disparity** structure: points at different
  depths map to different positions along their epipolar lines

> **Practical use:** In real VO systems, the epipolar constraint is used to
> *filter* putative feature matches — any match whose point does not lie within
> a few pixels of the predicted epipolar line is rejected as an outlier. This
> is far cheaper than full RANSAC and serves as a powerful pre-filter.

---

## 2. 8-Point Algorithm

The 8-point algorithm is the workhorse behind real-time relative pose estimation on resource-constrained drone and robot platforms, where its closed-form SVD solution avoids expensive iterative solvers.

### From Epipolar Constraint to Linear System

Given $N$ correspondences in **normalized** image coordinates
$(\hat{\mathbf{x}}_1^{(i)},\, \hat{\mathbf{x}}_2^{(i)})$, each pair satisfies:

$$\hat{\mathbf{x}}_2^{(i)\top}\, E\, \hat{\mathbf{x}}_1^{(i)} = 0$$

Writing $\hat{\mathbf{x}}_1 = (\hat{x}_1, \hat{y}_1, 1)^\top$ and
$\hat{\mathbf{x}}_2 = (\hat{x}_2, \hat{y}_2, 1)^\top$, the constraint expands to:

$$\hat{x}_2 \hat{x}_1 e_{11} + \hat{x}_2 \hat{y}_1 e_{12} + \hat{x}_2 e_{13} + \hat{y}_2 \hat{x}_1 e_{21} + \hat{y}_2 \hat{y}_1 e_{22} + \hat{y}_2 e_{23} + \hat{x}_1 e_{31} + \hat{y}_1 e_{32} + e_{33} = 0$$

### Vectorization

Flatten $E$ into a 9-vector:

$$\mathbf{e} = [e_{11}, e_{12}, e_{13}, e_{21}, e_{22}, e_{23}, e_{31}, e_{32}, e_{33}]^\top$$

Each correspondence gives one linear equation $\mathbf{a}_i^\top \mathbf{e} = 0$ where:

$$\mathbf{a}_i = [\hat{x}_1\hat{x}_2,\; \hat{y}_1\hat{x}_2,\; \hat{x}_2,\; \hat{x}_1\hat{y}_2,\; \hat{y}_1\hat{y}_2,\; \hat{y}_2,\; \hat{x}_1,\; \hat{y}_1,\; 1]$$

Stacking $N$ rows gives the $N \times 9$ system:

$$A\,\mathbf{e} = \mathbf{0}$$

### SVD Solution

This is a **homogeneous** linear system.  We seek the non-trivial $\mathbf{e}$ that minimizes
$\|A\mathbf{e}\|^2$ subject to $\|\mathbf{e}\| = 1$.  The solution is the right singular
vector corresponding to the **smallest** singular value — the **last column of $V$** in:

$$A = U \Sigma V^\top$$

With $N \geq 8$ correspondences and $\text{rank}(A) = 8$, the solution is unique up to sign.

> **Why the last column of $V$?** We want to minimize $\|A\mathbf{e}\|$ subject to
> $\|\mathbf{e}\| = 1$ (otherwise the trivial $\mathbf{e} = \mathbf{0}$ is a solution).
> By the SVD $A = U\Sigma V^\top$, we have
> $\|A\mathbf{e}\|^2 = \|\Sigma V^\top \mathbf{e}\|^2$.
> Setting $\mathbf{g} = V^\top \mathbf{e}$ (an orthogonal change of variables preserving
> $\|\mathbf{g}\| = 1$), we minimize $\sum_i \sigma_i^2 g_i^2$, which is minimized when
> $\mathbf{g} = \mathbf{e}_n$ (the last standard basis vector, corresponding to the
> smallest singular value $\sigma_n$). Therefore $\mathbf{e} = V\mathbf{e}_n$, the last
> column of $V$.

### Hartley Normalization

The raw 8-point algorithm is **numerically unstable** because image coordinates can range
from 0 to several hundred pixels.  Hartley's isotropic normalization rescales the points to
have **zero mean** and **average distance** $\sqrt{2}$ from the origin:

$$T = \begin{bmatrix} s & 0 & -s\bar{x} \\ 0 & s & -s\bar{y} \\ 0 & 0 & 1 \end{bmatrix}, \qquad s = \frac{\sqrt{2}}{\text{mean\_dist}}$$

where $\bar{x}, \bar{y}$ is the centroid and mean\_dist is the average distance of the
points from the centroid.

> **Why normalization matters:** Without normalization, the entries of $A$ span many orders
> of magnitude (pixel coordinates ~100–1000 vs. the homogeneous coordinate = 1). This causes
> the smallest singular values of $A$ to be dominated by numerical noise rather than geometric
> constraints, making the null-space vector unreliable. Hartley & Zisserman (MVG, §11.2)
> showed that normalizing points to have zero centroid and $\sqrt{2}$ average distance from
> the origin equalizes the scale of the problem, improving the condition number of $A$ by
> several orders of magnitude.

**Algorithm:**
1. Compute normalization transforms $T_1, T_2$ for each point set
2. Normalize: $\tilde{\mathbf{x}}_i = T_i \hat{\mathbf{x}}_i$
3. Build $A$ from normalized points and solve the SVD problem → $\tilde{E}$
4. Denormalize: $E = T_2^\top \tilde{E}\, T_1$

### Rank-2 Projection

The SVD solution generally does not have rank 2.  We project onto the manifold of valid
essential matrices:

1. SVD: $E = U\,\text{diag}(\sigma_1, \sigma_2, \sigma_3)\, V^\top$
2. Replace: $E \leftarrow U\,\text{diag}\!\left(\tfrac{\sigma_1+\sigma_2}{2},\; \tfrac{\sigma_1+\sigma_2}{2},\; 0\right) V^\top$

This is the closest rank-2 matrix with two equal singular values in Frobenius norm.

In [ ]:
def normalize_points(pts):
    """Hartley normalization: zero mean, average distance sqrt(2).
    pts: (N, 2)
    Returns: (N, 3) homogeneous normalized points, (3, 3) transform T.
    """
    centroid = np.mean(pts, axis=0)
    shifted = pts - centroid
    mean_dist = np.mean(np.linalg.norm(shifted, axis=1))
    s = np.sqrt(2) / (mean_dist + 1e-12)

    T = np.array([
        [s, 0, -s * centroid[0]],
        [0, s, -s * centroid[1]],
        [0, 0, 1]
    ])

    pts_h = np.hstack([pts, np.ones((len(pts), 1))])
    pts_norm = (T @ pts_h.T).T
    return pts_norm, T

In [ ]:
def eight_point_algorithm(pts1, pts2):
    """Normalized 8-point algorithm for Essential Matrix estimation.
    pts1, pts2: (N, 2) normalized image coordinates.
    Returns: (3, 3) essential matrix with ||E||_F = sqrt(2).
    """
    pts1_norm, T1 = normalize_points(pts1)
    pts2_norm, T2 = normalize_points(pts2)

    N = len(pts1)
    A = np.zeros((N, 9))
    for i in range(N):
        x1, y1 = pts1_norm[i, 0], pts1_norm[i, 1]
        x2, y2 = pts2_norm[i, 0], pts2_norm[i, 1]
        A[i] = [x1*x2, y1*x2, x2, x1*y2, y1*y2, y2, x1, y1, 1]

    _, _, Vt = np.linalg.svd(A)
    E_hat = Vt[-1].reshape(3, 3)

    E_hat = T2.T @ E_hat @ T1

    U_e, S_e, Vt_e = np.linalg.svd(E_hat)
    s_avg = (S_e[0] + S_e[1]) / 2.0
    E_proj = U_e @ np.diag([s_avg, s_avg, 0.0]) @ Vt_e

    E_proj = E_proj / np.linalg.norm(E_proj) * np.sqrt(2)
    return E_proj

In [ ]:
np.random.seed(42)
pts_3d_8pt = generate_3d_points(n=100)
K = make_K()

R1_8pt, t1_8pt = np.eye(3), np.zeros(3)
R2_8pt = rotation_matrix([0, 1, 0], np.deg2rad(8))
t2_8pt = np.array([0.3, 0.0, 0.05])

px1_clean = project(pts_3d_8pt, K, R1_8pt, t1_8pt)
px2_clean = project(pts_3d_8pt, K, R2_8pt, t2_8pt)

K_inv = np.linalg.inv(K)
n1_clean = (K_inv @ np.hstack([px1_clean, np.ones((100, 1))]).T).T[:, :2]
n2_clean = (K_inv @ np.hstack([px2_clean, np.ones((100, 1))]).T).T[:, :2]

E_est_clean = eight_point_algorithm(n1_clean, n2_clean)
E_true_norm = skew(t2_8pt) @ R2_8pt
E_true_norm = E_true_norm / np.linalg.norm(E_true_norm) * np.sqrt(2)

print("=== Noise-free test ===")
print(f"E estimated:\n{E_est_clean}")
print(f"\nE ground truth (normalized):\n{E_true_norm}")

diff = min(np.linalg.norm(E_est_clean - E_true_norm),
           np.linalg.norm(E_est_clean + E_true_norm))
print(f"\nDifference: {diff:.2e} (should be ≈ 0 for noise-free)")

In [ ]:
np.random.seed(42)
noise_sigma = 0.5
px1_noisy = px1_clean + np.random.randn(*px1_clean.shape) * noise_sigma
px2_noisy = px2_clean + np.random.randn(*px2_clean.shape) * noise_sigma

n1_noisy = (K_inv @ np.hstack([px1_noisy, np.ones((100, 1))]).T).T[:, :2]
n2_noisy = (K_inv @ np.hstack([px2_noisy, np.ones((100, 1))]).T).T[:, :2]

E_est_noisy = eight_point_algorithm(n1_noisy, n2_noisy)

print(f"=== With noise (σ = {noise_sigma} px) ===")
print(f"E estimated:\n{E_est_noisy}")

U_check, S_check, _ = np.linalg.svd(E_est_noisy)
print(f"\nSingular values: {S_check}")
print(f"(Should be [σ, σ, 0] after rank-2 projection)")

### 5-Point Algorithm (Nistér, 2004)

The 8-point algorithm estimates $E$ from $\geq 8$ correspondences, but it solves for a
general $3 \times 3$ matrix ($9 - 1 = 8$ DOF up to scale) and only enforces the essential
matrix constraints *a posteriori* via rank-2 projection.  Since calibrated cameras give us
normalized coordinates directly, we can work with $E$ instead of $F$ — and $E$ has only
**5 DOF** (3 for rotation + 2 for translation direction, since $\|\mathbf{t}\|$ is
unrecoverable from epipolar geometry).

**Why 5 points suffice.**  Each correspondence provides one scalar constraint
$\hat{\mathbf{x}}_2^\top E\, \hat{\mathbf{x}}_1 = 0$.  Five correspondences give
5 equations in the 9 entries of $E$, leaving a 4-dimensional null space.  The algebraic
constraints on $E$ — the **cubic** constraint $\det(E) = 0$ and the **trace** constraint

$$2\,E\,E^\top E - \text{tr}(E\,E^\top)\,E = 0$$

(equivalent to requiring two equal nonzero singular values) — supply the remaining equations.

**Deriving the trace constraint.** Since $E = [\mathbf{t}]_\times R$ and $[\mathbf{t}]_\times^\top [\mathbf{t}]_\times = \|\mathbf{t}\|^2 I - \mathbf{t}\mathbf{t}^\top$ (proved in Notebook 05), we have $E^\top E = R^\top(\|\mathbf{t}\|^2 I - \mathbf{t}\mathbf{t}^\top)R$. This matrix has eigenvalues $(\|\mathbf{t}\|^2, \|\mathbf{t}\|^2, 0)$ with trace $\text{tr}(E^\top E) = 2\|\mathbf{t}\|^2$. Writing the SVD as $E = U\,\text{diag}(\sigma, \sigma, 0)\,V^\top$, compute:

$$EE^\top E = U\,\text{diag}(\sigma^3, \sigma^3, 0)\,V^\top, \qquad \text{tr}(EE^\top) = 2\sigma^2$$

Therefore $2EE^\top E = U\,\text{diag}(2\sigma^3, 2\sigma^3, 0)\,V^\top = 2\sigma^2 \cdot E = \text{tr}(EE^\top) E$, confirming the identity. This gives **9 cubic equations** in the entries of $E$ (one per matrix element), but only 4 are independent due to symmetry.

Substituting the 4-parameter family from the null space into these constraints yields a
system of polynomial equations with **up to 10 solutions** over $\mathbb{C}$.

**Practical solvers.**  Nistér's original algorithm uses a clever elimination to reduce the
system to a $10^\text{th}$-degree polynomial in one unknown.  Later work (Stewénius et al.,
2006) formulates it as a **Gröbner basis** problem; Li & Hartley (2006) cast it as a
**polynomial eigenvalue** problem.  All produce the (at most) 10 candidate $E$ matrices,
from which the geometrically valid solution is selected by cheirality checks.

**Why 5-point is preferred in practice.**  In a RANSAC loop, the number of iterations to
find an outlier-free sample scales as

$$k = \frac{\log(1 - p)}{\log\!\bigl(1 - w^n\bigr)}$$

where $w$ is the inlier ratio, $n$ is the sample size, and $p$ is the desired success
probability.  Reducing $n$ from 8 to 5 dramatically cuts $k$ — e.g.\ at $w = 0.5$,
$k$ drops from $1177$ to $145$ (for $p = 0.99$).

> **Note:** `cv2.findEssentialMat` uses the 5-point solver internally (with RANSAC or
> LMEDS), making it both faster and more robust than an 8-point + RANSAC pipeline.

### RANSAC: Making Estimation Robust to Outliers

In real images, feature matching produces **outliers** — incorrect correspondences
caused by repeated textures, occlusion, or descriptor ambiguity. Even a small fraction
of outliers (5–20%) can completely corrupt the essential matrix estimate. RANSAC
(Random Sample Consensus, Fischler & Bolles, 1981) is the standard solution.

**Algorithm:**

1. **Sample** a minimal set of $n$ correspondences ($n = 5$ for the 5-point algorithm,
   $n = 8$ for the 8-point algorithm)
2. **Estimate** the essential matrix $E$ from this minimal set
3. **Score** by counting inliers: correspondences whose epipolar error
   $|\mathbf{x}_2^\top F \mathbf{x}_1|$ is below a threshold $\tau$
4. **Repeat** for $k$ iterations, keeping the best $E$
5. **Refine** the final $E$ using all inliers (not just the minimal set)

**How many iterations?** The number of RANSAC iterations needed to find at least one
outlier-free sample with probability $p$ is:

$$k = \frac{\log(1 - p)}{\log(1 - w^n)}$$

where $w$ is the inlier ratio and $n$ is the sample size. This formula explains why
the **5-point algorithm is strongly preferred** over the 8-point algorithm in RANSAC:

| Inlier ratio $w$ | 8-point ($n{=}8$) | 5-point ($n{=}5$) | Speedup |
|:-:|:-:|:-:|:-:|
| 70% | 78 iterations | 17 iterations | $4.6\times$ |
| 50% | 1177 iterations | 145 iterations | $8.1\times$ |
| 30% | 49k iterations | 3.3k iterations | $15\times$ |

At 50% inlier ratio (common in challenging conditions), the 5-point algorithm needs
$8\times$ fewer iterations — a critical advantage for real-time VO.

**Modern RANSAC variants:**
- **PROSAC** (Chum & Matas, 2005): samples high-quality matches first (sorted by
  descriptor distance), converging faster than uniform sampling
- **MAGSAC++** (Barath et al., 2020): eliminates the inlier threshold $\tau$ via
  marginalizing over a range of noise levels
- **GC-RANSAC** (Barath & Matas, 2018): adds a local optimization step that
  refines the model using graph-cut-based inlier selection

> `cv2.findEssentialMat` uses the 5-point algorithm inside a RANSAC loop by
> default, making it both statistically optimal and outlier-robust.

---

## 3. Essential Matrix Decomposition

Correctly decomposing $E$ into rotation and translation is what allows a drone or robot to know which direction it moved between frames — a wrong decomposition means flying or driving in the wrong direction.

### Extracting $(R, \mathbf{t})$ from $E$

Given $E = [\mathbf{t}]_\times R$ and its SVD $E = U \Sigma V^\top$ with
$\Sigma = \text{diag}(\sigma, \sigma, 0)$, we want to recover $R$ and $\mathbf{t}$.

### Key Factorization Matrices

Define the orthogonal matrix $W$ (a $90°$ rotation about the $z$-axis) and the
skew-symmetric matrix $Z$:

$$W = \begin{bmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & 1 \end{bmatrix}, \qquad Z = \begin{bmatrix} 0 & 1 & 0 \\ -1 & 0 & 0 \\ 0 & 0 & 0 \end{bmatrix}$$

Note that $Z = \text{diag}(1,1,0) \cdot W^\top$ (verify: $W^\top = \begin{bsmallmatrix}0&1&0\\-1&0&0\\0&0&1\end{bsmallmatrix}$, zeroing the last row gives $Z$).

### The Four Solutions — Derivation

**Why does $E = U \Sigma V^\top$ decompose into $R = UWV^\top$ and $\mathbf{t} = \mathbf{u}_3$?**

The essential matrix satisfies $E = [\mathbf{t}]_\times R$, where $[\mathbf{t}]_\times$
is skew-symmetric (rank 2) and $R$ is orthogonal. Given $E = U\,\text{diag}(\sigma, \sigma, 0)\,V^\top$,
define:

$$
W = \begin{pmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & 1 \end{pmatrix}
$$

Note that $\text{diag}(1,1,0) \cdot W^\top$ is skew-symmetric. Setting $Z = \text{diag}(1,1,0) \cdot W^\top$, we can write:

$$
E = U\,\sigma\,(Z \cdot W)\,V^\top
= \sigma\,\underbrace{(U Z U^\top)}_{[\mathbf{t}]_\times}\;
  \underbrace{(U W V^\top)}_{R}
$$

Since $UZU^\top$ is skew-symmetric and has the same rank structure, its generating
vector is $\pm \sigma \mathbf{u}_3$ (the third column of $U$ scaled by $\sigma$).
Since $E$ is only defined up to scale, $\mathbf{t} = \pm \mathbf{u}_3$.

> **Proof that $UZU^\top$ is skew-symmetric:** Since $Z = \text{diag}(1,1,0) \cdot W^\top$ is skew-symmetric (verify: $Z^\top = W \cdot \text{diag}(1,1,0) = -Z$), and $U$ is orthogonal, we have $(UZU^\top)^\top = UZ^\top U^\top = U(-Z)U^\top = -(UZU^\top)$. So $UZU^\top$ is skew-symmetric.
>
> **Why $\mathbf{t} = \pm \mathbf{u}_3$:** Any 3×3 skew-symmetric matrix $S$ has the form $S = [\mathbf{v}]_\times$ for some vector $\mathbf{v}$, and its null space is spanned by $\mathbf{v}$ (since $[\mathbf{v}]_\times \mathbf{v} = \mathbf{v} \times \mathbf{v} = 0$). For $UZU^\top$: its null vector is $U \cdot \text{null}(Z) = U \mathbf{e}_3 = \mathbf{u}_3$ (since $Z$ has a zero in the (3,3) position). Therefore the generating vector of $UZU^\top$ is $\pm \sigma \mathbf{u}_3$, giving $\mathbf{t} = \pm \mathbf{u}_3$ (up to the scale $\sigma$ which we absorb into the overall scale ambiguity of $E$).

Using $W$ gives one rotation; using $W^\top$ gives another. Combined with the
$\pm$ sign of $\mathbf{t}$, this yields four solutions:

| Solution | Rotation | Translation |
|:--------:|:--------:|:-----------:|
| 1 | $R = UWV^\top$ | $\mathbf{t} = +\mathbf{u}_3$ |
| 2 | $R = UWV^\top$ | $\mathbf{t} = -\mathbf{u}_3$ |
| 3 | $R = UW^\top V^\top$ | $\mathbf{t} = +\mathbf{u}_3$ |
| 4 | $R = UW^\top V^\top$ | $\mathbf{t} = -\mathbf{u}_3$ |

where $\mathbf{u}_3$ is the **third column** of $U$.

**Important:** We must ensure $\det(R) = +1$ (a proper rotation, not a reflection).
If $\det(R) = -1$, negate both $R$ and $\mathbf{t}$.

### Cheirality Check

Only **one** of the four solutions places all observed 3D points in front of **both** cameras.
For each candidate $(R, \mathbf{t})$:

1. Triangulate a point $\mathbf{X}$ from a correspondence
2. Check $Z_1 > 0$: the point is in front of camera 1
3. Transform to camera 2: $\mathbf{X}_2 = R\mathbf{X} + \mathbf{t}$
4. Check $Z_2 > 0$: the point is in front of camera 2

The four solutions correspond geometrically to:
- **Correct**: point in front of both cameras ✓
- **Reflected about cam 1**: point behind camera 1
- **Reflected about cam 2**: point behind camera 2
- **Reflected about both**: point behind both cameras

In [ ]:
def decompose_essential(E):
    """Decompose E into 4 candidate (R, t) solutions."""
    U, S, Vt = np.linalg.svd(E)

    if np.linalg.det(U) < 0:
        U = -U
    if np.linalg.det(Vt) < 0:
        Vt = -Vt

    W = np.array([[0, -1, 0],
                  [1,  0, 0],
                  [0,  0, 1]], dtype=np.float64)

    R1 = U @ W @ Vt
    R2 = U @ W.T @ Vt
    t = U[:, 2]

    solutions = [
        (R1, +t),
        (R1, -t),
        (R2, +t),
        (R2, -t),
    ]

    for i, (R, t_) in enumerate(solutions):
        if np.linalg.det(R) < 0:
            solutions[i] = (-R, -t_)

    return solutions

In [ ]:
def triangulate_dlt(p1, p2, P1, P2):
    """DLT triangulation for a single point pair.
    p1, p2: (3,) homogeneous image coords (pixels).
    P1, P2: (3, 4) projection matrices.
    Returns: (3,) 3D point.
    """
    A = np.array([
        p1[0] * P1[2] - P1[0],
        p1[1] * P1[2] - P1[1],
        p2[0] * P2[2] - P2[0],
        p2[1] * P2[2] - P2[1],
    ])
    _, _, Vt = np.linalg.svd(A)
    X = Vt[-1]
    return X[:3] / X[3]


def cheirality_check(R, t, pts1_norm, pts2_norm, K):
    """Count points that are in front of both cameras."""
    P1 = np.hstack([np.eye(3), np.zeros((3, 1))])
    P2 = np.hstack([R, t.reshape(3, 1)])

    count = 0
    for i in range(len(pts1_norm)):
        p1_h = np.array([pts1_norm[i, 0], pts1_norm[i, 1], 1.0])
        p2_h = np.array([pts2_norm[i, 0], pts2_norm[i, 1], 1.0])
        X = triangulate_dlt(p1_h, p2_h, P1, P2)
        X2 = R @ X + t
        if X[2] > 0 and X2[2] > 0:
            count += 1
    return count

In [ ]:
solutions = decompose_essential(E_est_noisy)

print("Four (R, t) solutions from E decomposition:\n")
for idx, (R_sol, t_sol) in enumerate(solutions):
    n_front = cheirality_check(R_sol, t_sol, n1_noisy, n2_noisy, K)
    print(f"Solution {idx+1}: det(R)={np.linalg.det(R_sol):+.4f}, "
          f"points in front of both cameras: {n_front}/100")

best_idx = np.argmax([cheirality_check(R, t, n1_noisy, n2_noisy, K)
                      for R, t in solutions])
R_best, t_best = solutions[best_idx]
print(f"\n→ Best solution: #{best_idx+1}")

In [ ]:
print("Recovered R:")
print(R_best)
print(f"\nGround truth R:")
print(R2_8pt)

R_err = R_best @ R2_8pt.T
angle_err = np.rad2deg(np.arccos(np.clip((np.trace(R_err) - 1) / 2, -1, 1)))
print(f"\nRotation error: {angle_err:.3f}°")

t_dir_est = t_best / np.linalg.norm(t_best)
t_dir_gt = t2_8pt / np.linalg.norm(t2_8pt)
t_angle = np.rad2deg(np.arccos(np.clip(np.abs(t_dir_est @ t_dir_gt), 0, 1)))
print(f"Translation direction error: {t_angle:.3f}°")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12),
                         subplot_kw={'projection': '3d'})
labels_4 = ['Correct (typical)', 'Behind cam 1', 'Behind cam 2', 'Behind both']

P1_vis = np.hstack([np.eye(3), np.zeros((3, 1))])

for ax, (R_sol, t_sol), label in zip(axes.flat, solutions, labels_4):
    P2_vis = np.hstack([R_sol, t_sol.reshape(3, 1)])

    pts_tri = []
    z_signs = []
    for i in range(min(50, len(n1_noisy))):
        p1_h = np.array([n1_noisy[i, 0], n1_noisy[i, 1], 1.0])
        p2_h = np.array([n2_noisy[i, 0], n2_noisy[i, 1], 1.0])
        X = triangulate_dlt(p1_h, p2_h, P1_vis, P2_vis)
        if np.linalg.norm(X) < 100:
            pts_tri.append(X)
            X2 = R_sol @ X + t_sol
            z_signs.append(X[2] > 0 and X2[2] > 0)

    pts_tri = np.array(pts_tri) if pts_tri else np.zeros((0, 3))

    if len(pts_tri) > 0:
        colors = ['green' if ok else 'red' for ok in z_signs]
        ax.scatter(pts_tri[:, 0], pts_tri[:, 2], pts_tri[:, 1],
                   c=colors, s=10, alpha=0.6)

    c2 = -R_sol.T @ t_sol
    ax.scatter(0, 0, 0, c='blue', s=80, marker='^', label='Cam 1')
    ax.scatter(c2[0], c2[2], c2[1], c='orange', s=80, marker='^', label='Cam 2')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('X'); ax.set_ylabel('Z'); ax.set_zlabel('Y')
    ax.view_init(elev=25, azim=-60)
    ax.legend(fontsize=8)

plt.suptitle('Four Solutions from Essential Matrix Decomposition\n'
             '(green = in front of both, red = behind one or both)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### Interpreting the Four Decomposition Solutions

The $2 \times 2$ grid above visualizes the geometric meaning of each solution
from the essential matrix decomposition:

**Why exactly four solutions?** The essential matrix encodes the epipolar
constraint $\mathbf{p}_2^\top E\, \mathbf{p}_1 = 0$, which is a *bilinear* form
in the two image points. This bilinearity means that $E$ cannot distinguish between
a point in front of the camera and its reflection behind it — both satisfy the
epipolar constraint identically. Combined with the $\pm$ ambiguity in translation
direction, this produces $2 \times 2 = 4$ candidate solutions.

**Cheirality as a physical constraint:** Of the four mathematically valid
decompositions, only one places all observed 3D points in front of *both* cameras
(positive depth in both camera frames). This is the **cheirality constraint** — a
physical requirement that cameras can only photograph points that reflect light
toward them.

**Color coding:** Green points satisfy the cheirality constraint (positive depth
in both cameras); red points violate it (negative depth in at least one camera).
The correct solution shows a cluster of green points in a physically plausible
configuration, while the three incorrect solutions show points "behind" one or
both cameras.

> **Implementation note:** In practice, it suffices to triangulate a single
> correspondence and check cheirality — if the point has positive depth in both
> cameras, that solution is correct. Testing multiple points improves robustness
> to noise. OpenCV's `cv2.recoverPose` does this automatically, returning only
> the geometrically valid $(R, \mathbf{t})$.

---

## 4. Triangulation

Triangulation turns 2D image matches into the 3D map that a robot uses for obstacle avoidance, path planning, and localization.

Given corresponding points in two calibrated views and the relative camera pose, we want to
reconstruct the 3D point.

### Method 1: Linear DLT Triangulation

From the projection equation $\lambda_i \mathbf{x}_i = P_i \mathbf{X}$ (with $\mathbf{X}$ in
homogeneous coordinates), we eliminate $\lambda_i$ via the cross product:

$$\mathbf{x}_i \times (P_i \mathbf{X}) = \mathbf{0}$$

Expanding with $\mathbf{x}_i = (x_i, y_i, 1)^\top$ and writing $P_i$ row-by-row as
$\mathbf{p}_i^{1\top}, \mathbf{p}_i^{2\top}, \mathbf{p}_i^{3\top}$, we get three equations
per view (2 independent):

$$\begin{aligned}
x_i\, \mathbf{p}_i^{3\top}\mathbf{X} - \mathbf{p}_i^{1\top}\mathbf{X} &= 0 \\
y_i\, \mathbf{p}_i^{3\top}\mathbf{X} - \mathbf{p}_i^{2\top}\mathbf{X} &= 0
\end{aligned}$$

Stacking the two views gives a $4 \times 4$ system $A\mathbf{X} = \mathbf{0}$, solved by SVD.

### Method 2: Midpoint Method

Model two rays from the camera centers through the image points:

$$\mathbf{r}_1(s) = \mathbf{o}_1 + s\,\mathbf{d}_1, \qquad \mathbf{r}_2(t) = \mathbf{o}_2 + t\,\mathbf{d}_2$$

Due to noise, the rays generally **don't intersect**.  Find $s^*, t^*$ minimizing the
squared distance between the rays:

$$\min_{s,t} \|\mathbf{r}_1(s) - \mathbf{r}_2(t)\|^2$$

Setting $\partial / \partial s = 0$ and $\partial / \partial t = 0$ gives the $2 \times 2$
linear system:

$$\begin{bmatrix} \mathbf{d}_1 \cdot \mathbf{d}_1 & -\mathbf{d}_1 \cdot \mathbf{d}_2 \\ \mathbf{d}_1 \cdot \mathbf{d}_2 & -\mathbf{d}_2 \cdot \mathbf{d}_2 \end{bmatrix} \begin{bmatrix} s \\ t \end{bmatrix} = \begin{bmatrix} -\mathbf{d}_1 \cdot (\mathbf{o}_1 - \mathbf{o}_2) \\ -\mathbf{d}_2 \cdot (\mathbf{o}_1 - \mathbf{o}_2) \end{bmatrix}$$

The triangulated 3D point is the **midpoint of closest approach**:

$$\mathbf{X} = \frac{\mathbf{r}_1(s^*) + \mathbf{r}_2(t^*)}{2}$$

### Method 3: Optimal (Hartley-Sturm)

DLT and midpoint are algebraically convenient but **statistically biased** — they don't
minimize a geometrically meaningful cost.  The optimal method minimizes the
**reprojection error**:

$$\min_{\mathbf{X}} \sum_{i=1}^{2} \|\mathbf{x}_i - \pi(P_i \mathbf{X})\|^2$$

Hartley and Sturm (MVG, Algorithm 12.1) show this can be solved via polynomial root-finding.
We implement an **iteratively reweighted DLT** approximation, which converges to the optimal
solution in a few iterations.

### Method 4: Lindström Triangulation

**Lindström (CVPR 2010, "Triangulation Made Easy")** achieves the same **optimal $L_2$
reprojection cost** as Hartley-Sturm but replaces the degree-6 polynomial root-finding with
a simple iterative correction scheme.

**Algorithm:**
1. Start with the DLT (linear) solution $\mathbf{X}^{(0)}$.
2. At each iteration $k$:
   - Compute the reprojection errors $\boldsymbol{\delta}_i^{(k)} = \mathbf{x}_i - \pi(P_i \mathbf{X}^{(k)})$
     in both views.
   - Solve a small **quadratic correction** to the observed image points:
     find corrected observations $\hat{\mathbf{x}}_i$ that lie exactly on the
     epipolar lines and are closest to the measurements in $L_2$.
   - Re-triangulate from $\hat{\mathbf{x}}_i$ using DLT → $\mathbf{X}^{(k+1)}$.
3. Converges in **2–3 iterations** to the global optimum.

**Advantages over Hartley-Sturm:**
- Simpler to implement — no polynomial root-finding or companion matrix eigenvalue computation.
- Same optimality guarantee (minimizes total squared reprojection error).
- Each iteration is a lightweight linear solve, making it very fast in practice.

> **Reference:** P. Lindström, "Triangulation Made Easy," *CVPR 2010*.

In [ ]:
def triangulate_midpoint(p1, p2, R1, t1, R2, t2):
    """Midpoint triangulation.
    p1, p2: (3,) normalized image coords.  R, t: world-to-camera.
    Returns: (3,) 3D point in world frame.
    """
    o1 = -R1.T @ t1
    o2 = -R2.T @ t2
    d1 = R1.T @ (p1 / np.linalg.norm(p1))
    d2 = R2.T @ (p2 / np.linalg.norm(p2))

    w = o1 - o2
    a = d1 @ d1
    b = d1 @ d2
    c = d2 @ d2
    d = d1 @ w
    e = d2 @ w

    denom = a * c - b * b
    if abs(denom) < 1e-12:
        return o1

    s = (b * e - c * d) / denom
    t_param = (a * e - b * d) / denom

    return (o1 + s * d1 + o2 + t_param * d2) / 2.0

In [ ]:
def triangulate_optimal(p1, p2, P1, P2, n_iter=10):
    """Iteratively reweighted DLT (approximating optimal triangulation).
    p1, p2: (3,) homogeneous pixel coords.  P1, P2: (3, 4).
    """
    w1, w2 = 1.0, 1.0

    for _ in range(n_iter):
        A = np.array([
            (p1[0] * P1[2] - P1[0]) / w1,
            (p1[1] * P1[2] - P1[1]) / w1,
            (p2[0] * P2[2] - P2[0]) / w2,
            (p2[1] * P2[2] - P2[1]) / w2,
        ])
        _, _, Vt = np.linalg.svd(A)
        X = Vt[-1]
        X = X / X[3]

        w1_new = P1[2] @ X
        w2_new = P2[2] @ X
        if abs(w1_new) < 1e-10 or abs(w2_new) < 1e-10:
            break
        w1, w2 = w1_new, w2_new

    return X[:3]

In [ ]:
np.random.seed(42)
pts_3d_tri = generate_3d_points(n=50, xlim=(-3, 3), ylim=(-2, 2), zlim=(5, 15))
K = make_K()

R1_tri, t1_tri = np.eye(3), np.zeros(3)
R2_tri = rotation_matrix([0, 1, 0], np.deg2rad(15))
t2_tri = np.array([1.0, 0.0, 0.0])

P1_tri = K @ np.hstack([R1_tri, t1_tri.reshape(3, 1)])
P2_tri = K @ np.hstack([R2_tri, t2_tri.reshape(3, 1)])

px1_c = project(pts_3d_tri, K, R1_tri, t1_tri)
px2_c = project(pts_3d_tri, K, R2_tri, t2_tri)

noise_levels = np.linspace(0, 3.0, 15)
errors_dlt, errors_mid, errors_opt = [], [], []

for sigma in noise_levels:
    np.random.seed(123)
    px1_n = px1_c + np.random.randn(*px1_c.shape) * sigma
    px2_n = px2_c + np.random.randn(*px2_c.shape) * sigma

    e_dlt, e_mid, e_opt = [], [], []
    K_inv_t = np.linalg.inv(K)

    for i in range(len(pts_3d_tri)):
        p1h = np.array([px1_n[i, 0], px1_n[i, 1], 1.0])
        p2h = np.array([px2_n[i, 0], px2_n[i, 1], 1.0])

        X_d = triangulate_dlt(p1h, p2h, P1_tri, P2_tri)
        e_dlt.append(np.linalg.norm(X_d - pts_3d_tri[i]))

        p1n = K_inv_t @ p1h
        p2n = K_inv_t @ p2h
        X_m = triangulate_midpoint(p1n, p2n, R1_tri, t1_tri, R2_tri, t2_tri)
        e_mid.append(np.linalg.norm(X_m - pts_3d_tri[i]))

        X_o = triangulate_optimal(p1h, p2h, P1_tri, P2_tri)
        e_opt.append(np.linalg.norm(X_o - pts_3d_tri[i]))

    errors_dlt.append(np.median(e_dlt))
    errors_mid.append(np.median(e_mid))
    errors_opt.append(np.median(e_opt))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(noise_levels, errors_dlt, 'o-', label='DLT', linewidth=2, markersize=5)
plt.plot(noise_levels, errors_mid, 's-', label='Midpoint', linewidth=2, markersize=5)
plt.plot(noise_levels, errors_opt, '^-', label='Optimal (iterative)', linewidth=2, markersize=5)
plt.xlabel('Noise σ (pixels)')
plt.ylabel('Median 3D Error')
plt.title('Triangulation Accuracy vs. Pixel Noise Level')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)
sigma_demo = 1.0
px1_demo = px1_c + np.random.randn(*px1_c.shape) * sigma_demo
px2_demo = px2_c + np.random.randn(*px2_c.shape) * sigma_demo

pts_recon = []
for i in range(len(pts_3d_tri)):
    p1h = np.array([px1_demo[i, 0], px1_demo[i, 1], 1.0])
    p2h = np.array([px2_demo[i, 0], px2_demo[i, 1], 1.0])
    pts_recon.append(triangulate_dlt(p1h, p2h, P1_tri, P2_tri))
pts_recon = np.array(pts_recon)

errs = np.linalg.norm(pts_recon - pts_3d_tri, axis=1)

reproj_errs = np.zeros(len(pts_recon))
for i in range(len(pts_recon)):
    p1_proj = project(pts_recon[i:i+1], K, R1_tri, t1_tri)[0]
    p2_proj = project(pts_recon[i:i+1], K, R2_tri, t2_tri)[0]
    reproj_errs[i] = 0.5 * (np.linalg.norm(p1_proj - px1_demo[i])
                             + np.linalg.norm(p2_proj - px2_demo[i]))

fig = plt.figure(figsize=(16, 12))

# ── 3D view: points colored by reprojection error ──
ax1 = fig.add_subplot(221, projection='3d')
sc = ax1.scatter(pts_recon[:, 0], pts_recon[:, 2], pts_recon[:, 1],
                 c=reproj_errs, cmap='RdYlGn_r', s=30, alpha=0.8,
                 edgecolors='none', label='Triangulated')
ax1.scatter(pts_3d_tri[:, 0], pts_3d_tri[:, 2], pts_3d_tri[:, 1],
            c='royalblue', s=15, alpha=0.4, marker='o', label='Ground Truth')
for i in range(len(pts_3d_tri)):
    ax1.plot([pts_3d_tri[i, 0], pts_recon[i, 0]],
             [pts_3d_tri[i, 2], pts_recon[i, 2]],
             [pts_3d_tri[i, 1], pts_recon[i, 1]],
             'gray', alpha=0.15, linewidth=0.4)

def _draw_cam_axes(ax, pos, R_cw, length=0.5):
    """Draw RGB camera axes (X=red, Y=green, Z=blue) in display coords (X,Z,Y)."""
    R_wc = R_cw.T
    for k, col in enumerate(['#e74c3c', '#2ecc71', '#3498db']):
        direction = R_wc[:, k] * length
        end = pos + direction
        ax.plot([pos[0], end[0]], [pos[2], end[2]], [pos[1], end[1]],
                '-', color=col, linewidth=2, alpha=0.9)

cam1_center = np.zeros(3)
cam2_center = -R2_tri.T @ t2_tri
_draw_cam_axes(ax1, cam1_center, R1_tri, length=0.8)
_draw_cam_axes(ax1, cam2_center, R2_tri, length=0.8)
ax1.scatter(*cam1_center[[0, 2, 1]], c='red', s=80, marker='^', zorder=5, label='Cam 1')
ax1.scatter(*cam2_center[[0, 2, 1]], c='green', s=80, marker='^', zorder=5, label='Cam 2')

cbar = plt.colorbar(sc, ax=ax1, shrink=0.5, pad=0.08)
cbar.set_label('Reproj. Error (px)', fontsize=9)
ax1.set_xlabel('X'); ax1.set_ylabel('Z'); ax1.set_zlabel('Y')
ax1.set_title('3D Point Cloud — Colored by Reprojection Error', fontsize=11)
ax1.legend(fontsize=7, loc='upper left')
ax1.view_init(elev=25, azim=-55)

# ── Top-down (XZ) view ──
ax2 = fig.add_subplot(222)
sc2 = ax2.scatter(pts_recon[:, 0], pts_recon[:, 2], c=reproj_errs,
                   cmap='RdYlGn_r', s=25, alpha=0.8, edgecolors='none')
ax2.scatter(pts_3d_tri[:, 0], pts_3d_tri[:, 2],
            c='royalblue', s=12, alpha=0.35, marker='o', label='GT')
ax2.scatter(cam1_center[0], cam1_center[2], c='red', s=100, marker='^',
            zorder=5, edgecolors='k', label='Cam 1')
ax2.scatter(cam2_center[0], cam2_center[2], c='green', s=100, marker='^',
            zorder=5, edgecolors='k', label='Cam 2')
plt.colorbar(sc2, ax=ax2, shrink=0.8, label='Reproj. Error (px)')
ax2.set_xlabel('X'); ax2.set_ylabel('Z')
ax2.set_title('Top-Down View (XZ)', fontsize=11)
ax2.legend(fontsize=8); ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

# ── Error distributions ──
ax3 = fig.add_subplot(223)
ax3.hist(errs, bins=20, color='steelblue', edgecolor='white', alpha=0.8, label='3D Error')
ax3.axvline(np.median(errs), color='red', linewidth=2, linestyle='--',
            label=f'Median = {np.median(errs):.3f}')
ax3.set_xlabel('3D Error (world units)'); ax3.set_ylabel('Count')
ax3.set_title('3D Triangulation Error Distribution', fontsize=11)
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

ax4 = fig.add_subplot(224)
ax4.hist(reproj_errs, bins=20, color='#e67e22', edgecolor='white', alpha=0.8,
         label='Reproj. Error')
ax4.axvline(np.median(reproj_errs), color='red', linewidth=2, linestyle='--',
            label=f'Median = {np.median(reproj_errs):.2f} px')
ax4.set_xlabel('Reprojection Error (px)'); ax4.set_ylabel('Count')
ax4.set_title('Reprojection Error Distribution', fontsize=11)
ax4.legend(fontsize=9); ax4.grid(True, alpha=0.3)

plt.suptitle(f'DLT Triangulation Quality (σ = {sigma_demo} px)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"3D error — mean: {np.mean(errs):.4f}, median: {np.median(errs):.4f}, "
      f"max: {np.max(errs):.4f}")
print(f"Reproj error — mean: {np.mean(reproj_errs):.2f} px, "
      f"median: {np.median(reproj_errs):.2f} px")

### Interpreting Triangulation Quality

The 4-panel visualization above provides a comprehensive view of DLT triangulation
performance:

**Reprojection error coloring (top-left):** Points colored by reprojection error
reveal a systematic pattern: points **far from the cameras** tend to have larger
errors because small angular errors in the viewing rays translate to large positional
errors at distance. This is the geometric dilution of precision (GDOP) effect —
the same phenomenon that affects GPS accuracy.

**Camera coordinate axes:** The RGB axes at each camera position (red = $X$, green
= $Y$, blue = $Z$) show the camera's local coordinate frame. In our setup, the
$Z$-axis (blue) points along the viewing direction. The angle between the two
cameras' $Z$-axes is the **convergence angle** — wider convergence generally yields
better triangulation accuracy (at the cost of fewer shared features).

**Top-down view (top-right):** This bird's-eye projection strips away vertical
information to reveal the lateral distribution of reconstruction errors. Points
near the baseline (line connecting the two cameras) are poorly conditioned because
the two viewing rays are nearly parallel — the "narrow triangle" problem.

**Error distributions (bottom row):** The histograms quantify what the scatter
plots show qualitatively:
- **3D error**: the Euclidean distance between triangulated and ground-truth
  positions. The long tail indicates a few poorly-conditioned points (typically
  far from the cameras or near the baseline).
- **Reprojection error**: the pixel-space residual after projecting the
  triangulated point back into both images. This is the cost function that
  optimal triangulation methods minimize.

> **Practical rule of thumb:** A triangulated point is considered reliable if its
> reprojection error is below 1 pixel and its depth is within $3\times$ the
> stereo baseline. Points failing these criteria should be rejected or downweighted
> in subsequent optimization.

---

## 5. Monocular VO Pipeline

### Pipeline Overview

For each consecutive frame pair $(I_{k-1}, I_k)$:

1. **Detect** features (ORB, SIFT, etc.)
2. **Match** features between frames
3. **Estimate** essential matrix $E$ with RANSAC
4. **Decompose** $E$ into $(R, \mathbf{t})$
5. **Triangulate** new 3D points
6. **Accumulate** pose: $T_{0 \to k} = T_{0 \to k-1} \cdot T_{k-1 \to k}$

### Pose Accumulation

`cv2.recoverPose` returns $(R, \mathbf{t})$ encoding the transform **from camera $k{-}1$ to camera $k$**: $\mathbf{x}_k = R\,\mathbf{x}_{k-1} + \mathbf{t}$. Let $T_{\text{rel}} = [R \mid \mathbf{t}]$ be this world-to-camera relative transform.

To accumulate **camera-to-world** poses (so that $T_{\text{accum}}[:3, 3]$ directly gives the camera position), we compose with the **inverse**:

$$T_{\text{cam}_k \to \text{world}} = T_{\text{cam}_{k-1} \to \text{world}} \cdot T_{\text{rel}}^{-1}$$

where $T_{\text{rel}}^{-1} = \begin{bmatrix} R^\top & -R^\top\mathbf{t} \\ \mathbf{0}^\top & 1 \end{bmatrix}$.

> **Convention note.** Equivalently, one can accumulate *world-to-camera* transforms: $T_{0 \to k} = T_{k-1 \to k} \cdot T_{0 \to k-1}$, then extract the camera position as $\mathbf{c}_k = -R_{0 \to k}^\top\, \mathbf{t}_{0 \to k}$. Both conventions give the same trajectory. Our code uses the camera-to-world convention for simplicity.

### Synthetic Data Generation

We generate a synthetic scene with:
- 3D point cloud: random points in a room-like box $[-5, 5]^2 \times [-5, 5]$
- Camera follows a **circular trajectory** of radius $r$, always looking at the origin
- Each camera pose projects the 3D points to pixel coordinates; Gaussian noise simulates
  feature detection/matching error

Trajectory equations:

$$x = r\cos\theta, \quad z = r\sin\theta, \quad y = 0$$

In [ ]:
def generate_circular_trajectory(n_frames=30, radius=5.0, height=0.0,
                                  target=np.array([0.0, 0.0, 0.0])):
    """Camera poses along a circular path looking at the target.
    Returns: list of (R, t) pairs (world-to-camera transforms).
    """
    poses = []
    angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)

    for theta in angles:
        cam_pos = np.array([radius * np.cos(theta),
                            height,
                            radius * np.sin(theta)])
        R, t = look_at(cam_pos, target)
        poses.append((R, t))

    return poses


def generate_figure8_trajectory(n_frames=60, radius=5.0, height=0.0,
                                 target=np.array([0.0, 0.0, 0.0])):
    """Figure-8 trajectory: x = r sin(θ), z = r sin(2θ)."""
    poses = []
    offset = 2 * np.pi / n_frames * 0.5
    angles = np.linspace(offset, 2 * np.pi + offset, n_frames, endpoint=False)

    for theta in angles:
        cam_pos = np.array([radius * np.sin(theta),
                            height,
                            radius * np.sin(2 * theta)])
        R, t = look_at(cam_pos, target)
        poses.append((R, t))

    return poses


def generate_spiral_trajectory(n_frames=50, radius=5.0, height_range=(-2, 2),
                                target=np.array([0.0, 0.0, 0.0])):
    """Spiral: circular path with linearly increasing height."""
    poses = []
    angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
    heights = np.linspace(height_range[0], height_range[1], n_frames)

    for theta, h in zip(angles, heights):
        cam_pos = np.array([radius * np.cos(theta), h, radius * np.sin(theta)])
        R, t = look_at(cam_pos, target)
        poses.append((R, t))

    return poses

In [ ]:
poses_circ = generate_circular_trajectory(40, radius=8)
poses_fig8 = generate_figure8_trajectory(60, radius=6)
poses_spir = generate_spiral_trajectory(50, radius=7)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), subplot_kw={'projection': '3d'})

for ax, poses, title in zip(axes,
    [poses_circ, poses_fig8, poses_spir],
    ['Circle', 'Figure-8', 'Spiral']):

    positions = np.array([-R.T @ t for R, t in poses])
    ax.plot(positions[:, 0], positions[:, 2], positions[:, 1],
            'b-o', markersize=2, linewidth=1.5)
    ax.scatter(positions[0, 0], positions[0, 2], positions[0, 1],
               c='green', s=80, marker='o', zorder=5, label='Start')
    ax.scatter(positions[-1, 0], positions[-1, 2], positions[-1, 1],
               c='red', s=80, marker='s', zorder=5, label='End')
    ax.scatter(0, 0, 0, c='gold', s=50, marker='*')
    ax.set_xlabel('X'); ax.set_ylabel('Z'); ax.set_zlabel('Y')
    ax.set_title(title)
    ax.view_init(elev=30, azim=-60)
    ax.legend(fontsize=8)

plt.suptitle('Synthetic Camera Trajectories', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
def run_monocular_vo(pts_3d, poses_gt, K, noise_sigma=1.0):
    """Run monocular VO pipeline on synthetic data.
    Uses ground-truth scale (from stereo/IMU in real systems).
    Returns: (est_positions, gt_positions) as (M, 3) arrays.
    """
    n_frames = len(poses_gt)
    gt_positions = np.array([-R.T @ t for R, t in poses_gt])

    R_first, t_first = poses_gt[0]
    T_accum = np.eye(4)
    T_accum[:3, :3] = R_first.T
    T_accum[:3, 3] = -R_first.T @ t_first
    est_positions = [T_accum[:3, 3].copy()]

    for i in range(1, n_frames):
        R_prev, t_prev = poses_gt[i - 1]
        R_curr, t_curr = poses_gt[i]

        px_prev = project(pts_3d, K, R_prev, t_prev)
        px_curr = project(pts_3d, K, R_curr, t_curr)

        vis_prev = ((px_prev[:, 0] >= 0) & (px_prev[:, 0] < 640) &
                    (px_prev[:, 1] >= 0) & (px_prev[:, 1] < 480))
        vis_curr = ((px_curr[:, 0] >= 0) & (px_curr[:, 0] < 640) &
                    (px_curr[:, 1] >= 0) & (px_curr[:, 1] < 480))

        in_front_p = ((R_prev @ pts_3d.T).T + t_prev)[:, 2] > 0.1
        in_front_c = ((R_curr @ pts_3d.T).T + t_curr)[:, 2] > 0.1

        vis = vis_prev & vis_curr & in_front_p & in_front_c

        if np.sum(vis) < 8:
            est_positions.append(est_positions[-1].copy())
            continue

        np.random.seed(i)
        px_p = px_prev[vis] + np.random.randn(np.sum(vis), 2) * noise_sigma
        px_c = px_curr[vis] + np.random.randn(np.sum(vis), 2) * noise_sigma

        E_cv, mask = cv2.findEssentialMat(
            px_p, px_c, K, method=cv2.RANSAC, prob=0.999, threshold=1.0)
        if E_cv is None:
            est_positions.append(est_positions[-1].copy())
            continue

        _, R_rel, t_rel, _ = cv2.recoverPose(E_cv, px_p, px_c, K)
        t_rel = t_rel.flatten()

        gt_scale = np.linalg.norm(-R_curr.T @ t_curr + R_prev.T @ t_prev)
        t_rel = t_rel * gt_scale

        T_rel = np.eye(4)
        T_rel[:3, :3] = R_rel
        T_rel[:3, 3] = t_rel

        T_accum = T_accum @ np.linalg.inv(T_rel)
        est_positions.append(T_accum[:3, 3].copy())

    return np.array(est_positions), gt_positions

In [ ]:
np.random.seed(42)
n_points_scene = 500
pts_3d_scene = generate_3d_points(n=n_points_scene,
                                   xlim=(-4, 4), ylim=(-3, 3), zlim=(-4, 4))
K = make_K()
n_frames_vo = 40
poses_gt = generate_circular_trajectory(n_frames=n_frames_vo, radius=8.0)

est_pos, gt_pos = run_monocular_vo(pts_3d_scene, poses_gt, K, noise_sigma=0.5)

print(f"Frames processed: {len(est_pos)}")
pos_err = np.linalg.norm(est_pos - gt_pos, axis=1)
print(f"Position error — mean: {np.mean(pos_err):.4f}, max: {np.max(pos_err):.4f}")

In [ ]:
drift = np.linalg.norm(est_pos - gt_pos, axis=1)
drift_max = drift.max() + 1e-12

def _draw_frustum(ax, pos, R_cw, scale=0.6, color='dodgerblue'):
    """Wireframe camera frustum at a given pose (display coords: X, Z, Y)."""
    hw, hh = 0.4 * scale, 0.3 * scale
    corners_cam = np.array([[-hw, -hh, scale],
                            [ hw, -hh, scale],
                            [ hw,  hh, scale],
                            [-hw,  hh, scale]])
    corners_w = (R_cw @ corners_cam.T).T + pos
    p = pos[[0, 2, 1]]
    cw = corners_w[:, [0, 2, 1]]
    for i in range(4):
        ax.plot([p[0], cw[i, 0]], [p[1], cw[i, 1]], [p[2], cw[i, 2]],
                '-', color=color, alpha=0.45, linewidth=0.8)
    for i in range(4):
        j = (i + 1) % 4
        ax.plot([cw[i, 0], cw[j, 0]], [cw[i, 1], cw[j, 1]], [cw[i, 2], cw[j, 2]],
                '-', color=color, alpha=0.45, linewidth=0.8)

fig = plt.figure(figsize=(18, 13))

# ── Panel 1: 3D view with drift-colored trajectory + camera frustums ──
ax1 = fig.add_subplot(221, projection='3d')
ax1.scatter(pts_3d_scene[:, 0], pts_3d_scene[:, 2], pts_3d_scene[:, 1],
            c='lightblue', s=2, alpha=0.12, rasterized=True)
ax1.plot(gt_pos[:, 0], gt_pos[:, 2], gt_pos[:, 1],
         '-', color='royalblue', linewidth=2, alpha=0.8, label='Ground Truth')

for j in range(len(est_pos) - 1):
    color = plt.cm.hot(drift[j] / drift_max)
    ax1.plot(est_pos[j:j+2, 0], est_pos[j:j+2, 2], est_pos[j:j+2, 1],
             '-', color=color, linewidth=2.5, solid_capstyle='round')

err_step = max(1, len(est_pos) // 8)
for j in range(0, len(est_pos), err_step):
    ax1.plot([gt_pos[j, 0], est_pos[j, 0]],
             [gt_pos[j, 2], est_pos[j, 2]],
             [gt_pos[j, 1], est_pos[j, 1]],
             'k--', alpha=0.45, linewidth=1)

frust_step = max(1, len(poses_gt) // 6)
for j in range(0, len(poses_gt), frust_step):
    R_cw_j = poses_gt[j][0]
    _draw_frustum(ax1, gt_pos[j], R_cw_j.T, scale=0.7)

ax1.scatter(*gt_pos[0, [0, 2, 1]], c='limegreen', s=100, marker='o',
            zorder=5, edgecolors='k', linewidths=0.8, label='Start')
ax1.scatter(*gt_pos[-1, [0, 2, 1]], c='darkred', s=100, marker='s',
            zorder=5, edgecolors='k', linewidths=0.8, label='End')

sm = plt.cm.ScalarMappable(cmap='hot', norm=plt.Normalize(0, drift_max))
sm.set_array([])
cbar1 = plt.colorbar(sm, ax=ax1, shrink=0.45, pad=0.08)
cbar1.set_label('Accumulated Drift', fontsize=9)

ax1.set_xlabel('X'); ax1.set_ylabel('Z'); ax1.set_zlabel('Y')
ax1.set_title('3D View — Trajectory Colored by Drift', fontsize=12)
ax1.view_init(elev=35, azim=-55)
ax1.legend(fontsize=8, loc='upper left')

# ── Panel 2: Top-down view with error arrows ──
ax2 = fig.add_subplot(222)
ax2.scatter(pts_3d_scene[:, 0], pts_3d_scene[:, 2],
            c='lightblue', s=2, alpha=0.12, rasterized=True)
ax2.plot(gt_pos[:, 0], gt_pos[:, 2],
         '-o', color='royalblue', markersize=3, linewidth=2, label='Ground Truth')
sc2 = ax2.scatter(est_pos[:, 0], est_pos[:, 2], c=drift, cmap='hot',
                   s=25, zorder=4, edgecolors='none')
ax2.plot(est_pos[:, 0], est_pos[:, 2], '-', color='gray', linewidth=0.7, alpha=0.5)
for j in range(0, len(est_pos), err_step):
    ax2.annotate('', xy=(est_pos[j, 0], est_pos[j, 2]),
                 xytext=(gt_pos[j, 0], gt_pos[j, 2]),
                 arrowprops=dict(arrowstyle='->', color='black', alpha=0.4, lw=1))
plt.colorbar(sc2, ax=ax2, shrink=0.8, label='Drift')
ax2.set_xlabel('X'); ax2.set_ylabel('Z')
ax2.set_title('Top-Down View (XZ) — Error Arrows', fontsize=12)
ax2.legend(fontsize=9); ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

# ── Panel 3: Per-axis (X, Y, Z) error over time ──
ax3 = fig.add_subplot(212)
err_xyz = est_pos - gt_pos
ax3.plot(err_xyz[:, 0], '-', color='#e74c3c', linewidth=1.5, label='$\\Delta X$')
ax3.plot(err_xyz[:, 1], '-', color='#2ecc71', linewidth=1.5, label='$\\Delta Y$')
ax3.plot(err_xyz[:, 2], '-', color='#3498db', linewidth=1.5, label='$\\Delta Z$')
ax3.fill_between(range(len(drift)), -drift, drift, alpha=0.08, color='gray',
                 label='$\\pm\\|\\mathbf{e}\\|$')
ax3.axhline(0, color='k', linewidth=0.5, alpha=0.3)
ax3.set_xlabel('Frame'); ax3.set_ylabel('Position Error (world units)')
ax3.set_title('Per-Axis Tracking Error Over Time', fontsize=12)
ax3.legend(fontsize=9, ncol=4, loc='upper left'); ax3.grid(True, alpha=0.3)

plt.suptitle('Monocular Visual Odometry — Circular Trajectory', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### Analyzing VO Drift Patterns

The rich visualization above reveals the anatomy of visual odometry drift. Several
observations are worth highlighting:

**Drift coloring (hot colormap):** The estimated trajectory transitions from cool
(low drift, early frames) to warm (high drift, later frames). This is a direct
visualization of the **fundamental limitation** of dead-reckoning systems: errors
accumulate monotonically because there is no global correction mechanism
(unlike SLAM, which uses loop closures to bound drift).

**Error vectors (dashed lines):** The arrows connecting ground truth to estimated
positions show both the **magnitude** and **direction** of drift. In a circular
trajectory, drift typically manifests as:
- **Tangential drift**: the estimated pose runs ahead of or behind the true position
  along the trajectory — caused by rotation estimation errors
- **Radial drift**: the estimated trajectory expands or shrinks relative to the
  true circle — caused by translation scale errors

**Camera frustums:** The wireframe pyramids show the field of view at key poses,
grounding the abstract trajectory in a physical camera model. The frustum orientation
encodes the camera's viewing direction — useful for verifying that the VO pipeline
correctly estimates rotation.

**Per-axis error decomposition:** The bottom panel separates total drift into its
$X$, $Y$, and $Z$ components. This reveals which **degrees of freedom** accumulate
error fastest. In a planar circular trajectory:
- $\Delta Y$ should remain near zero (no vertical motion)
- $\Delta X$ and $\Delta Z$ may show correlated oscillations from rotation error
- The grey envelope $\pm\|\mathbf{e}\|$ bounds the per-component errors and
  shows the overall drift magnitude

**Practical implications:** This analysis directly informs system design:
- If tangential drift dominates → improve rotation estimation (use gyroscope fusion)
- If radial drift dominates → improve scale estimation (use stereo or depth)
- If vertical drift appears → check IMU gravity alignment or feature distribution

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(pos_err, 'r-', linewidth=2)
plt.xlabel('Frame')
plt.ylabel('Position Error (world units)')
plt.title('Accumulated VO Position Error Over Time')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 6. Scale Ambiguity

Scale ambiguity is why a monocular-only drone cannot determine whether it has moved 1 m or 10 m, making fusion with IMU or stereo essential for safe autonomous flight.

### The Fundamental Problem

The essential matrix $E = [\mathbf{t}]_\times R$ encodes relative rotation and translation,
but $E$ is defined only **up to scale**: for any $\alpha > 0$,

$$\alpha E = [\alpha\mathbf{t}]_\times R$$

So from $E$ alone we recover only the **direction** of $\mathbf{t}$, not its magnitude.

### Proof of Scale Ambiguity

If $(R, \mathbf{t}, \mathbf{X})$ is a consistent solution (projections match the observed
image points), then $(R, \alpha\mathbf{t}, \alpha\mathbf{X})$ is also consistent for any
$\alpha > 0$.

The projection of $\alpha\mathbf{X}$ under pose $(R, \alpha\mathbf{t})$:

$$K(R(\alpha\mathbf{X}) + \alpha\mathbf{t}) = \alpha\, K(R\mathbf{X} + \mathbf{t})$$

After projective normalization (dividing by the $z$-component), the scalar $\alpha$
**cancels**.  The projected 2D points are identical — scale is **unobservable** from images.

### Consequences

- Monocular VO produces trajectories with **correct shape** but **unknown scale**
- Scale can **drift** over long sequences because each inter-frame scale is independently
  estimated and errors accumulate

### Solutions to the Scale Problem

| Method | How it provides scale |
|--------|----------------------|
| **Stereo VO** | Known baseline between stereo cameras gives absolute scale |
| **Visual-Inertial** | IMU accelerometer measures metric acceleration |
| **Known objects** | Known physical size of objects in the scene |
| **Depth sensors** | LiDAR or RGB-D cameras provide metric depth |

In [ ]:
np.random.seed(42)
poses_s = generate_circular_trajectory(n_frames=30, radius=8.0)
gt_pos_s = np.array([-R.T @ t for R, t in poses_s])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

scales = [0.5, 1.0, 2.0, 5.0]
colors = ['red', 'blue', 'green', 'purple']

for s, c in zip(scales, colors):
    sp = gt_pos_s * s
    axes[0].plot(sp[:, 0], sp[:, 2], '-o', markersize=3, color=c,
                linewidth=1.5, label=f'Scale = {s}')

axes[0].set_xlabel('X'); axes[0].set_ylabel('Z')
axes[0].set_title('Same Trajectory, Different Scales')
axes[0].legend(); axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

In [ ]:
n_drift = 80
poses_drift = generate_circular_trajectory(n_frames=n_drift, radius=8.0)
gt_pos_drift = np.array([-R.T @ t for R, t in poses_drift])

np.random.seed(99)
scale_noise = 1.0 + np.cumsum(np.random.randn(n_drift) * 0.015)
scale_noise[0] = 1.0

drifted = np.zeros_like(gt_pos_drift)
drifted[0] = gt_pos_drift[0]
for i in range(1, n_drift):
    delta = gt_pos_drift[i] - gt_pos_drift[i - 1]
    drifted[i] = drifted[i - 1] + delta * scale_noise[i]

corrected = drifted.copy()
for i in range(1, n_drift):
    corrected[i] = drifted[0] + (drifted[i] - drifted[0]) / scale_noise[i]

fig = plt.figure(figsize=(18, 10))

# ── Panel 1: 3D trajectory comparison ──
ax1 = fig.add_subplot(221, projection='3d')
ax1.plot(gt_pos_drift[:, 0], gt_pos_drift[:, 2], gt_pos_drift[:, 1],
         '-', color='royalblue', linewidth=2, label='Ground Truth')
ax1.plot(drifted[:, 0], drifted[:, 2], drifted[:, 1],
         '--', color='#e74c3c', linewidth=1.8, label='With Scale Drift')
ax1.plot(corrected[:, 0], corrected[:, 2], corrected[:, 1],
         '-.', color='#2ecc71', linewidth=1.8, label='After Correction')
ax1.set_xlabel('X'); ax1.set_ylabel('Z'); ax1.set_zlabel('Y')
ax1.set_title('3D Trajectory: Before & After Scale Correction')
ax1.legend(fontsize=8); ax1.view_init(elev=30, azim=-55)

# ── Panel 2: Top-down overlay ──
ax2 = fig.add_subplot(222)
ax2.plot(gt_pos_drift[:, 0], gt_pos_drift[:, 2],
         '-o', color='royalblue', markersize=2, linewidth=2, label='Ground Truth')
ax2.plot(drifted[:, 0], drifted[:, 2],
         '--x', color='#e74c3c', markersize=2, linewidth=1.5, label='Drifted')
ax2.plot(corrected[:, 0], corrected[:, 2],
         '-.s', color='#2ecc71', markersize=2, linewidth=1.5, label='Corrected')
ax2.set_xlabel('X'); ax2.set_ylabel('Z')
ax2.set_title('Top-Down: Before & After Scale Correction')
ax2.legend(fontsize=9); ax2.set_aspect('equal'); ax2.grid(True, alpha=0.3)

# ── Panel 3: Per-frame scale factor ──
ax3 = fig.add_subplot(223)
ax3.plot(scale_noise, '-', color='#e74c3c', linewidth=2, label='Scale factor')
ax3.axhline(1.0, color='royalblue', linewidth=1.5, linestyle='--',
            alpha=0.7, label='Ideal (1.0)')
ax3.fill_between(range(n_drift), 1.0, scale_noise, alpha=0.15, color='red')
ax3.set_xlabel('Frame'); ax3.set_ylabel('Scale Factor')
ax3.set_title('Per-Frame Scale Factor Over Time')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

# ── Panel 4: Scale error percentage ──
ax4 = fig.add_subplot(224)
scale_err_pct = np.abs(scale_noise - 1.0) * 100
ax4.plot(scale_err_pct, '-', color='#e74c3c', linewidth=2)
ax4.fill_between(range(n_drift), 0, scale_err_pct, alpha=0.15, color='red')
ax4.set_xlabel('Frame'); ax4.set_ylabel('Scale Error (%)')
ax4.set_title('Cumulative Scale Drift (% deviation from true scale)')
ax4.grid(True, alpha=0.3)

plt.suptitle('Scale Drift in Monocular VO', fontsize=15)
plt.tight_layout()
plt.show()

### Understanding Scale Drift in Practice

The visualization above reveals several important insights about monocular scale drift:

**Why the trajectory spirals:** Each inter-frame translation is estimated with a
slightly wrong scale factor. These per-frame scale errors *compound multiplicatively*:
if the scale estimate is systematically biased (even by a few percent), the trajectory
gradually expands or contracts relative to the ground truth, producing the characteristic
spiral pattern.

**Scale error is non-stationary:** The bottom-left plot shows the per-frame scale
factor wandering away from 1.0 over time. This is a **random walk** in scale space —
the standard deviation of the scale error grows as $O(\sqrt{n})$ with the number
of frames $n$, which is why monocular VO becomes less reliable over long sequences.

**Scale correction strategies used in practice:**

| Strategy | How it works | Latency | Accuracy |
|----------|-------------|---------|----------|
| **Stereo baseline** | Known distance between cameras gives absolute scale per frame | Zero | High |
| **IMU preintegration** | Double-integrate accelerometer → metric displacement | Low (IMU rate) | Medium (drift without vision) |
| **Known objects** | Detect objects of known size (e.g. cars, lane width) | Variable | Medium |
| **GPS/GNSS** | Absolute position at low rate → scale correction | High (1–10 Hz) | Medium (multipath) |
| **Depth foundation models** | Learned monocular depth (e.g. Depth Anything) | Per-frame | Medium–High |

The **before/after correction** overlay in the top-right panel shows that simple
post-hoc scale correction (dividing out the estimated drift) can substantially
recover the true trajectory shape — but this requires knowing the true scale, which
defeats the purpose. In practice, the strategies above provide scale *during*
operation, not after the fact.

> **Key takeaway:** Scale ambiguity is not a bug — it is a **fundamental property**
> of projective geometry. Monocular VO systems that claim metric output always rely
> on an external scale source, whether they make it explicit or not.

---

## 7. PnP for Stereo VO

PnP is how autonomous vehicles localize against a pre-built 3D map — it directly gives metric pose, enabling safe navigation without the scale ambiguity of monocular VO.

### The PnP Problem

Given $N$ known 3D-to-2D correspondences $(\mathbf{X}_i, \mathbf{x}_i)$, find the camera
pose $(R, \mathbf{t})$ such that:

$$\lambda_i\, \mathbf{x}_i = K\,[R \mid \mathbf{t}]\, \mathbf{X}_i$$

Unlike the essential matrix approach, PnP directly gives **metric** pose when the 3D points
have metric scale (e.g., from stereo triangulation).

### P3P: Minimal Case

With 3 correspondences we get the **minimal solver** P3P.  Given three 3D points
$A, B, C$ and their normalized image projections $\hat{a}, \hat{b}, \hat{c}$:

1. Compute pairwise 3D distances: $d_{AB} = \|A - B\|$, $d_{BC}$, $d_{AC}$
2. Compute angles between bearing vectors from the camera center:
   $$\cos\alpha = \hat{b} \cdot \hat{c}, \qquad \cos\beta = \hat{a} \cdot \hat{c}, \qquad \cos\gamma = \hat{a} \cdot \hat{b}$$
3. Apply the **law of cosines** in the triangles formed by the camera center $O$ and each pair of 3D points. With $s_a = \|O - A\|$, $s_b = \|O - B\|$, $s_c = \|O - C\|$:

$$s_b^2 + s_c^2 - 2 s_b s_c \cos\alpha = d_{BC}^2$$
$$s_a^2 + s_c^2 - 2 s_a s_c \cos\beta = d_{AC}^2$$
$$s_a^2 + s_b^2 - 2 s_a s_b \cos\gamma = d_{AB}^2$$

4. Substitute $u = s_a/s_c$, $v = s_b/s_c$ to eliminate $s_c$, reducing to 2 equations in 2 unknowns. After elimination of one variable, this yields a **degree-4 polynomial** in the remaining unknown, giving up to **4 real solutions**. Each solution determines $(s_a, s_b, s_c)$ and hence the 3D point positions in the camera frame, from which $R, \mathbf{t}$ follow by Procrustes alignment.

### EPnP: Efficient PnP

EPnP represents the $N$ 3D points as weighted sums of **4 virtual control points**:

$$\mathbf{X}_i = \sum_{j=1}^{4} \alpha_{ij}\, \mathbf{c}_j, \qquad \sum_j \alpha_{ij} = 1$$

The camera pose is determined by finding the 3D control-point positions in the camera frame.
This has $O(N)$ complexity and is the standard in most SLAM/VO systems.

### Iterative PnP (Levenberg-Marquardt)

Minimizes reprojection error directly:

$$\min_{R,\mathbf{t}} \sum_{i=1}^{N} \|\mathbf{x}_i - \pi(K(R\mathbf{X}_i + \mathbf{t}))\|^2$$

This is refined with LM (see Section 9).

### Key Advantage Over Essential Matrix

If the 3D map points have **metric scale** (from stereo or depth sensor), the recovered
pose from PnP also has metric scale.  **No scale ambiguity!**

In [ ]:
def run_pnp_vo(pts_3d, poses_gt, K, noise_sigma=1.0):
    """PnP-based VO pipeline (assumes known 3D points with metric scale)."""
    n_frames = len(poses_gt)
    gt_positions = np.array([-R.T @ t for R, t in poses_gt])
    est_positions = [gt_positions[0].copy()]

    for i in range(1, n_frames):
        R_curr, t_curr = poses_gt[i]
        px_curr = project(pts_3d, K, R_curr, t_curr)

        vis = ((px_curr[:, 0] >= 0) & (px_curr[:, 0] < 640) &
               (px_curr[:, 1] >= 0) & (px_curr[:, 1] < 480))
        in_front = ((R_curr @ pts_3d.T).T + t_curr)[:, 2] > 0.1
        vis = vis & in_front

        if np.sum(vis) < 6:
            est_positions.append(est_positions[-1].copy())
            continue

        np.random.seed(i + 1000)
        pts_2d = px_curr[vis] + np.random.randn(np.sum(vis), 2) * noise_sigma
        pts_3d_vis = pts_3d[vis]

        ok, rvec, tvec, inl = cv2.solvePnPRansac(
            pts_3d_vis.astype(np.float64),
            pts_2d.astype(np.float64),
            K.astype(np.float64), None,
            iterationsCount=200, reprojectionError=2.0,
            flags=cv2.SOLVEPNP_ITERATIVE)

        if not ok:
            est_positions.append(est_positions[-1].copy())
            continue

        R_est, _ = cv2.Rodrigues(rvec)
        cam_pos = -R_est.T @ tvec.flatten()
        est_positions.append(cam_pos)

    return np.array(est_positions), gt_positions

In [ ]:
np.random.seed(42)
pts_3d_cmp = generate_3d_points(n=500, xlim=(-4, 4), ylim=(-3, 3), zlim=(-4, 4))
K = make_K()
poses_cmp = generate_circular_trajectory(n_frames=50, radius=8.0)

np.random.seed(42)
est_mono, gt_mono = run_monocular_vo(pts_3d_cmp, poses_cmp, K, noise_sigma=0.8)

np.random.seed(42)
est_pnp, gt_pnp = run_pnp_vo(pts_3d_cmp, poses_cmp, K, noise_sigma=0.8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot(gt_mono[:, 0], gt_mono[:, 2], 'b-o', markersize=3, label='Ground Truth')
axes[0].plot(est_mono[:, 0], est_mono[:, 2], 'r-x', markersize=3, label='Mono VO')
axes[0].set_title('Monocular VO (Essential Matrix)')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Z')
axes[0].legend(); axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)

axes[1].plot(gt_pnp[:, 0], gt_pnp[:, 2], 'b-o', markersize=3, label='Ground Truth')
axes[1].plot(est_pnp[:, 0], est_pnp[:, 2], 'g-^', markersize=3, label='PnP VO')
axes[1].set_title('PnP VO (Known 3D Points)')
axes[1].set_xlabel('X'); axes[1].set_ylabel('Z')
axes[1].legend(); axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)

plt.suptitle('Monocular VO vs PnP VO', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
err_mono_q = np.linalg.norm(est_mono - gt_mono, axis=1)
err_pnp_q = np.linalg.norm(est_pnp - gt_pnp, axis=1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(err_mono_q, 'r-', linewidth=2, label=f'Mono VO (mean={np.mean(err_mono_q):.3f})')
ax.plot(err_pnp_q, 'g-', linewidth=2, label=f'PnP VO  (mean={np.mean(err_pnp_q):.3f})')
ax.set_xlabel('Frame')
ax.set_ylabel('Position Error')
ax.set_title('Per-Frame Position Error: Mono VO vs PnP VO')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Choosing Your VO Pipeline: Mono vs. PnP

The results above illustrate a fundamental trade-off in VO system design:

**Monocular VO (Essential Matrix)**
- Requires only a single camera — the cheapest and lightest sensor package
- Suffers from **scale ambiguity**: translation is recovered only up to an unknown
  positive factor, so position error can grow unboundedly
- **Best for:** initial prototyping, weight-constrained systems (nano-drones),
  or when fused with IMU for scale (Visual-Inertial Odometry)

**PnP-based VO (Known 3D Structure)**
- Requires a pre-existing 3D map or a stereo/depth sensor to triangulate landmarks
  with metric scale
- Recovers **metric** poses directly — no scale ambiguity, much lower drift
- **Best for:** autonomous driving (stereo cameras), AR/VR with depth sensors,
  relocalization against a known map

**Hybrid pipeline (typical in practice):**

$$\underbrace{\text{Initialize with } E}_{\text{2D-2D, first frames}}
\;\to\;
\underbrace{\text{Triangulate landmarks}}_{\text{build local map}}
\;\to\;
\underbrace{\text{Track with PnP}}_{\text{3D-2D, metric scale}}
\;\to\;
\underbrace{\text{Refine with BA}}_{\text{joint optimization}}$$

This is precisely how systems like ORB-SLAM3 operate: essential matrix
initialization → triangulation → PnP tracking → local/global bundle adjustment.
The error plots above quantify *why* this hybrid design dominates: the 2D-2D
initialization gets the system started, but 3D-2D tracking (PnP) keeps drift in check.

---

## 8. Learned VO (Discussion)

### Evolution of Visual Odometry

| Era | Approach | Key Idea |
|-----|----------|----------|
| Classical | Feature-based | Hand-crafted features (ORB, SIFT) + geometric solvers |
| Classical | Direct | Minimize photometric error on raw pixels (DSO, LSD-SLAM) |
| Hybrid | Learned front-end | Learned features (SuperPoint) + classical back-end |
| End-to-end | Fully learned | Neural network estimates pose directly |

### DPVO (Deep Patch Visual Odometry)

**Teed & Deng, 2024** — A patch-based approach combining strengths of direct and
feature-based methods.

**Key ideas:**
- Track small image **patches** (not sparse keypoints) across frames
- **Differentiable bundle adjustment** layer that optimizes poses and patch depths jointly
- **Recurrent update operator** (inspired by RAFT optical flow) iteratively refines
  patch correspondences and depths
- Real-time capable (~15–25 FPS on modern GPU)

Architecture flow:

$$\text{Images} \xrightarrow{\text{CNN}} \text{Features} \xrightarrow{\text{Patches}} \text{Correlation} \xrightarrow{\text{GRU}} (\Delta\text{pose},\, \Delta\text{depth})$$

### DPV-SLAM

**Lipson et al., 2024** — Extends DPVO to a full SLAM system with:

- **Loop closure detection** via learned place recognition
- **Global optimization** to eliminate accumulated drift when loops are closed
- **Map reuse** — can localize against a previously built map

### DROID-SLAM

**Teed & Deng, 2021** — State-of-the-art learned SLAM.

**Key innovations:**
- **Dense optical flow** between all frame pairs in a sliding window
- **Differentiable Dense Bundle Adjustment (DBA)** — solves for poses and dense depth by
  minimizing reprojection error induced by predicted flow
- **Dense Recurrent Optical-flow Updates (DROID)** — recurrent network that iteratively
  refines optical flow
- State-of-the-art on TartanAir, EuRoC, TUM-RGBD benchmarks

### FoundationSLAM (AAAI 2026)

**FoundationSLAM** represents the emerging trend of plugging **depth foundation models**
into classical SLAM pipelines as geometric priors.

**Key ideas:**
- Uses a pretrained depth foundation model (**Depth Anything**) to produce dense monocular
  depth estimates, which are then used to **guide optical flow estimation** for dense
  monocular SLAM.
- Depth priors dramatically improve flow accuracy in **textureless regions** and under
  **large motions**, where purely appearance-based flow networks struggle.
- Achieves **real-time performance at 18 FPS**, bridging the gap between heavyweight
  learned SLAM systems (e.g.\ DROID-SLAM at ~5 FPS) and classical pipelines.

**Significance:**  Rather than training an end-to-end SLAM network from scratch,
FoundationSLAM treats the depth foundation model as a **plug-in prior** — a modular
component that can be swapped as better foundation models become available.  This
"foundation model as prior" paradigm is likely to become a dominant design pattern as
large-scale pretrained vision models continue to improve.

### Comparison Table

| Criterion | Classical VO | DPVO | DROID-SLAM | FoundationSLAM |
|-----------|:------------|:-----|:-----------|:---------------|
| **Accuracy** | Good (well-textured) | Very good | State-of-the-art | Very good |
| **Speed** | Real-time | Real-time | ~5 FPS | 18 FPS |
| **Generalization** | Hand-tuned per domain | Moderate | Moderate | Strong (foundation prior) |
| **Failure modes** | Textureless, motion blur | Domain shift | GPU memory, domain shift | Depth model quality |
| **Training data** | None needed | Large-scale video | Large-scale video | Pretrained depth model |
| **Loop closure** | Needs BoW/DBoW2 | No (VO only) | Yes | No |
| **Dense output** | No (sparse) | No (patches) | Yes (dense depth) | Yes (dense depth) |

### When to Use Which?

- **Classical VO**: Well-textured environments, good calibration, limited compute,
  need for interpretability and certifiability (robotics, aerospace)
- **Learned VO (DPVO)**: Real-time with better robustness than classical, can tolerate
  coarser depth
- **Learned SLAM (DROID)**: Maximum accuracy, GPU available, dense reconstruction desired,
  challenging visual conditions
- **Foundation-guided (FoundationSLAM)**: Real-time dense monocular SLAM, especially in
  textureless or challenging scenes where depth priors help most

### The Open Question: Are Classical Features Dead?

The explosion of learned VO/SLAM systems in 2024–2026 raises a fundamental question for
the field.

**The case against classical features:**
- **DROID-SLAM** (Teed & Deng, 2021): learned dense optical flow + classical BA backbone
  achieves state-of-the-art accuracy *without any hand-crafted feature detector*
- **DPVO / DPV-SLAM** (Lipson et al., 2024): sparse **patch** tracking achieves comparable
  accuracy to dense methods at real-time speeds — proving that dense computation is not
  strictly necessary
- **Foundation-3D systems**: MASt3R-SLAM and VGGT-SLAM 2.0 bypass the classical pipeline
  entirely — they produce dense RGB point cloud maps, don't require camera calibration
  (the model predicts it internally), and VGGT-SLAM 2.0 runs in real-time on Jetson Thor
  with open-set object detection

**The case for classical features:**
- **Efficiency**: ORB features run at >100 FPS on a Raspberry Pi; learned features need
  a GPU
- **Interpretability**: each step of the classical pipeline is inspectable and debuggable
- **Certifiability**: safety-critical applications (aerospace, autonomous driving) require
  provable guarantees that neural networks cannot yet provide
- **Generalization**: classical features work on *any* camera without domain-specific
  training data

**The emerging consensus (2026):** The future is likely **hybrid** — learned components
handle perception (depth estimation, optical flow, feature matching) while classical
optimization handles inference (bundle adjustment, pose graph optimization, loop closure).
This gives the best of both worlds: neural robustness with geometric rigor.

> **Reference:** Luo, Y. et al., "Why does Deep Learning Improve Visual SLAM?",
> *arXiv 2607.06023*, July 2026 — a systematic analysis of where learned components
> outperform classical modules, and where they don't.

---

## Bundle Adjustment and Local Optimization

Bundle adjustment (BA) is the gold standard for refining both camera poses and 3D
structure simultaneously. Understanding its mathematical structure is essential for
grasping why modern SLAM systems scale to thousands of keyframes.

### BA as Nonlinear Least Squares

BA jointly optimizes $M$ camera poses $\{T_i\}_{i=1}^M$ and $N$ 3D landmark positions
$\{\mathbf{X}_j\}_{j=1}^N$ by minimizing the sum of squared reprojection errors:

$$\min_{\{T_i\}, \{\mathbf{X}_j\}} \sum_{(i,j) \in \mathcal{O}} \rho\!\left( \left\| \pi(T_i, \mathbf{X}_j) - \mathbf{z}_{ij} \right\|^2_{\Sigma_{ij}} \right)$$

where:
- $\mathbf{z}_{ij}$ is the observed 2D projection of landmark $j$ in camera $i$
- $\pi(T_i, \mathbf{X}_j)$ is the predicted projection under pose $T_i$
- $\Sigma_{ij}$ is the measurement covariance (often isotropic: $\sigma^2 I$)
- $\rho(\cdot)$ is a robust kernel (Huber, Cauchy) to downweight outliers
- $\mathcal{O}$ is the set of observations (which camera sees which landmark)

### Why BA is Sparse

The key computational insight: **each observation involves only one camera and one
point**. The Jacobian $J$ of the stacked residual vector has a very sparse block
structure — most entries are zero because point $j$ does not appear in camera $i$
unless $(i,j) \in \mathcal{O}$.

The Hessian approximation $H = J^\top J$ inherits a characteristic
**arrow-head** sparsity pattern. Partitioning the state into camera parameters
$\mathbf{c}$ and structure $\mathbf{s}$:

$$H = \begin{bmatrix} H_{cc} & H_{cs} \\ H_{sc} & H_{ss} \end{bmatrix}$$

$H_{cc}$ is block-diagonal ($M$ blocks of $6 \times 6$) and $H_{ss}$ is
block-diagonal ($N$ blocks of $3 \times 3$). This sparsity is what makes
the **Schur complement trick** (derived in §9 below) so powerful — eliminating
$\Delta\mathbf{s}$ reduces the system from $(6M + 3N) \times (6M + 3N)$ to just
$6M \times 6M$.

### Sliding-Window BA for Bounded Complexity

Full BA over the entire trajectory grows without bound. Practical VO systems use
**sliding-window BA**: optimize only the last $W$ keyframes (typically $W = 5$–$20$)
and their associated landmarks, marginalizing out older poses:

$$\min_{\{T_{k-W+1}, \ldots, T_k\},\; \{\mathbf{X}_j\}} \;\sum \rho(\|r_{ij}\|^2) \;+\; \underbrace{\|\mathbf{r}_{\text{prior}}\|^2_{\Lambda}}_{\text{marginalization prior}}$$

The marginalization prior $\|\mathbf{r}_{\text{prior}}\|^2_{\Lambda}$ encodes
information from older frames that have been removed from the optimization window,
preventing information loss. This keeps per-frame computation bounded at
$O(W^2 N_W)$ rather than $O(k^2 N)$.

> **Reference:** Triggs, B. et al., "Bundle Adjustment — A Modern Synthesis,"
> *Vision Algorithms: Theory and Practice*, LNCS 1883, Springer, 2000 — the
> definitive survey of BA theory, sparsity exploitation, and gauge freedom.

---

## 9. Nonlinear Least Squares for Pose Refinement

Every production SLAM system on drones and self-driving cars refines its pose estimates with Gauss-Newton or LM — the difference between a raw estimate and a refined one can be the difference between a safe landing and a crash.

### Problem Formulation

Given an initial pose estimate $(R_0, \mathbf{t}_0)$ from essential matrix decomposition
or PnP, we **refine** it by minimizing the **reprojection error**:

$$\min_{R, \mathbf{t}} \sum_{i=1}^{N} \|\mathbf{x}_i - \pi(R\,\mathbf{X}_i + \mathbf{t})\|^2$$

where
$\pi(\mathbf{P}) = \begin{bmatrix} f_x P_x / P_z + c_x \\ f_y P_y / P_z + c_y \end{bmatrix}$
is the projection function.

### Parameterization via Lie Algebra

We cannot directly optimize over $R$ (rotation matrices are constrained to $\text{SO}(3)$).
Instead, parameterize small pose updates with
$\boldsymbol{\xi} = [\delta t_x, \delta t_y, \delta t_z, \delta\phi_x, \delta\phi_y, \delta\phi_z]^\top \in \mathbb{R}^6$.

The rotation update uses the axis-angle exponential:

$$R' = \exp([\boldsymbol{\delta\phi}]_\times) \cdot R, \qquad \mathbf{t}' = \mathbf{t} + \boldsymbol{\delta t}$$

For small $\boldsymbol{\delta\phi}$, $\exp([\boldsymbol{\delta\phi}]_\times) \approx I + [\boldsymbol{\delta\phi}]_\times$.

### Deriving the Jacobian

The residual for point $i$:

$$\mathbf{r}_i = \mathbf{x}_i - \pi(R\,\mathbf{X}_i + \mathbf{t})$$

Let $\mathbf{P}_i = R\,\mathbf{X}_i + \mathbf{t} = (P_x, P_y, P_z)^\top$.  The Jacobian of
$\pi$ with respect to $\mathbf{P}$:

$$\frac{\partial \pi}{\partial \mathbf{P}} = \begin{bmatrix} f_x / P_z & 0 & -f_x P_x / P_z^2 \\ 0 & f_y / P_z & -f_y P_y / P_z^2 \end{bmatrix}$$

The Jacobian of $\mathbf{P}$ with respect to pose parameters $\boldsymbol{\xi}$:

$$\frac{\partial \mathbf{P}}{\partial \boldsymbol{\xi}} = \begin{bmatrix} I_{3 \times 3} & -[\mathbf{P}]_\times \end{bmatrix} \in \mathbb{R}^{3 \times 6}$$

> **Convention note:** This Jacobian is derived from the SE(3) left perturbation 
> $T' = \exp(\hat{\delta\xi}) \cdot T$, which gives $\delta P = \delta\rho - [P]_\times \delta\phi$
> and thus $\partial P/\partial\delta\xi = [I_3 \mid -[P]_\times]$. In practice, the camera pose
> update uses the simpler decoupled rule $R' = \exp([\delta\phi]_\times) R$ and 
> $\mathbf{t}' = \mathbf{t} + \delta\mathbf{t}$. The decoupled Jacobian is
> $[I_3 \mid -[R\mathbf{X}]_\times]$, differing from the left-perturbation form
> $[I_3 \mid -[P]_\times]$ by $[\mathbf{t}]_\times \delta\phi$ — a **first-order** term in
> $\delta\phi$, but one that is small when $\|\mathbf{t}\| \ll \|P\|$ and in any case
> is compensated by subsequent Gauss-Newton iterations. This mixing of conventions is
> standard practice in systems like g2o, GTSAM, and ORB-SLAM.

By the chain rule:

$$J_i = -\frac{\partial \pi}{\partial \mathbf{P}} \cdot \frac{\partial \mathbf{P}}{\partial \boldsymbol{\xi}} \in \mathbb{R}^{2 \times 6}$$

### Gauss-Newton Update

Stack all residuals $\mathbf{r} = [\mathbf{r}_1^\top, \ldots, \mathbf{r}_N^\top]^\top$ and
Jacobians $J = [J_1^\top, \ldots, J_N^\top]^\top$.  The **normal equations**:

$$(J^\top J)\,\Delta\boldsymbol{\xi} = -J^\top \mathbf{r}$$

Update: $\boldsymbol{\xi} \leftarrow \boldsymbol{\xi} + \Delta\boldsymbol{\xi}$, then apply
to $R$ and $\mathbf{t}$.

### Levenberg-Marquardt Damping

Add a damping term $\lambda I$ to the Hessian approximation:

$$(J^\top J + \lambda I)\,\Delta\boldsymbol{\xi} = -J^\top \mathbf{r}$$

| $\lambda$ | Behavior |
|-----------|----------|
| Large | $\Delta\boldsymbol{\xi} \approx -\frac{1}{\lambda} J^\top \mathbf{r}$ — gradient descent (safe) |
| Small | ≈ Gauss-Newton (fast near minimum) |

**Adaptive strategy:** Decrease $\lambda$ when cost decreases (trust GN more); increase
$\lambda$ when cost increases (back off to gradient descent).

### Connection to Bundle Adjustment & the Schur Complement

BA jointly optimizes all camera poses $\{R_k, \mathbf{t}_k\}$ and 3D points $\{\mathbf{X}_i\}$. Partition the state into **camera parameters** $\mathbf{c}$ and **structure** $\mathbf{s}$. The normal equations become block-structured:

$$
\begin{bmatrix} H_{cc} & H_{cs} \\ H_{sc} & H_{ss} \end{bmatrix}
\begin{bmatrix} \Delta\mathbf{c} \\ \Delta\mathbf{s} \end{bmatrix}
= -\begin{bmatrix} \mathbf{g}_c \\ \mathbf{g}_s \end{bmatrix}
$$

where $H_{cc} = J_c^\top J_c$ is block-diagonal ($M$ cameras, each 6 DOF), and $H_{ss} = J_s^\top J_s$ is block-diagonal ($N$ points, each 3 DOF). The key insight: **$H_{ss}$ is trivially invertible** (it's block-diagonal with $3\times 3$ blocks). Eliminating $\Delta\mathbf{s}$ via the **Schur complement**:

$$\underbrace{(H_{cc} - H_{cs} H_{ss}^{-1} H_{sc})}_{S}\,\Delta\mathbf{c} = -(\mathbf{g}_c - H_{cs} H_{ss}^{-1} \mathbf{g}_s)$$

The **reduced camera system** $S$ is only $6M \times 6M$ (vs. the full $(6M + 3N) \times (6M + 3N)$). Since $N \gg M$ typically (thousands of points, tens of cameras), this is dramatically cheaper. After solving for $\Delta\mathbf{c}$, back-substitute:

$$\Delta\mathbf{s} = H_{ss}^{-1}(-\mathbf{g}_s - H_{sc}\,\Delta\mathbf{c})$$

**Complexity:** Full system: $O((6M+3N)^3)$. Schur complement: $O(M^3 + MN)$.

In [ ]:
def project_point(P, fx, fy, cx, cy):
    """Project a single camera-frame 3D point to pixel coords."""
    return np.array([fx * P[0] / P[2] + cx,
                     fy * P[1] / P[2] + cy])


def compute_residuals_and_jacobian(pts_3d, pts_2d, R, t, K):
    """Compute reprojection residuals and the full Jacobian."""
    fx, fy, cx, cy = K[0, 0], K[1, 1], K[0, 2], K[1, 2]
    N = len(pts_3d)
    residuals = np.zeros(2 * N)
    J = np.zeros((2 * N, 6))

    for i in range(N):
        P = R @ pts_3d[i] + t
        Px, Py, Pz = P

        proj = project_point(P, fx, fy, cx, cy)
        residuals[2*i:2*i+2] = pts_2d[i] - proj

        dpi_dP = np.array([
            [fx / Pz, 0,       -fx * Px / Pz**2],
            [0,       fy / Pz, -fy * Py / Pz**2]
        ])

        dP_dxi = np.hstack([np.eye(3), -skew(P)])  # (3, 6)

        J[2*i:2*i+2, :] = -dpi_dP @ dP_dxi

    return residuals, J

In [ ]:
def gauss_newton_pose(pts_3d, pts_2d, R_init, t_init, K,
                       n_iter=30, use_lm=False, lam0=1e-3):
    """Gauss-Newton (or Levenberg-Marquardt) pose refinement.
    Returns: refined R, t, and list of cost values per iteration.
    """
    R = R_init.copy()
    t = t_init.copy()
    lam = lam0
    costs = []

    for _ in range(n_iter):
        r, J = compute_residuals_and_jacobian(pts_3d, pts_2d, R, t, K)
        cost = 0.5 * np.sum(r**2)
        costs.append(cost)

        JtJ = J.T @ J
        Jtr = J.T @ r

        if use_lm:
            delta = np.linalg.solve(JtJ + lam * np.eye(6), -Jtr)
            angle = np.linalg.norm(delta[3:6])
            R_new = (rotation_matrix(delta[3:6], angle) if angle > 1e-15
                     else np.eye(3)) @ R
            t_new = t + delta[:3]

            r_new, _ = compute_residuals_and_jacobian(pts_3d, pts_2d, R_new, t_new, K)
            cost_new = 0.5 * np.sum(r_new**2)

            if cost_new < cost:
                R, t = R_new, t_new
                lam = max(lam * 0.1, 1e-12)
            else:
                lam = min(lam * 10.0, 1e8)
        else:
            delta = np.linalg.solve(JtJ + 1e-8 * np.eye(6), -Jtr)
            t = t + delta[:3]
            angle = np.linalg.norm(delta[3:6])
            if angle > 1e-15:
                R = rotation_matrix(delta[3:6] / angle, angle) @ R

    r_final, _ = compute_residuals_and_jacobian(pts_3d, pts_2d, R, t, K)
    costs.append(0.5 * np.sum(r_final**2))

    return R, t, costs

In [ ]:
np.random.seed(42)
K = make_K()

pts_3d_gn = generate_3d_points(n=80, xlim=(-3, 3), ylim=(-2, 2), zlim=(5, 15))

R_true_gn = rotation_matrix([0.1, 1.0, 0.2], np.deg2rad(15))
t_true_gn = np.array([0.5, -0.3, 0.2])

pts_2d_gn = project(pts_3d_gn, K, R_true_gn, t_true_gn)
pts_2d_gn += np.random.randn(*pts_2d_gn.shape) * 0.5

noise_R = rotation_matrix([0.3, -0.2, 0.5], np.deg2rad(5))
R_init_gn = noise_R @ R_true_gn
t_init_gn = t_true_gn + np.random.randn(3) * 0.3

print(f"Initial rotation error (Frobenius): {np.linalg.norm(R_init_gn - R_true_gn):.4f}")
print(f"Initial translation error: {np.linalg.norm(t_init_gn - t_true_gn):.4f}")

In [ ]:
R_gn, t_gn, costs_gn = gauss_newton_pose(
    pts_3d_gn, pts_2d_gn, R_init_gn, t_init_gn, K, n_iter=30, use_lm=False)

R_lm, t_lm, costs_lm = gauss_newton_pose(
    pts_3d_gn, pts_2d_gn, R_init_gn, t_init_gn, K, n_iter=30, use_lm=True, lam0=1e-2)

print("=== Gauss-Newton ===")
print(f"  R error (Frob): {np.linalg.norm(R_gn - R_true_gn):.6f}")
print(f"  t error:        {np.linalg.norm(t_gn - t_true_gn):.6f}")
print(f"  Final cost:     {costs_gn[-1]:.4f}")

print("\n=== Levenberg-Marquardt ===")
print(f"  R error (Frob): {np.linalg.norm(R_lm - R_true_gn):.6f}")
print(f"  t error:        {np.linalg.norm(t_lm - t_true_gn):.6f}")
print(f"  Final cost:     {costs_lm[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(costs_gn, 'b-o', markersize=4, linewidth=2, label='Gauss-Newton')
axes[0].semilogy(costs_lm, 'r-s', markersize=4, linewidth=2, label='Levenberg-Marquardt')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cost (log scale)')
axes[0].set_title('Convergence of Pose Refinement')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

r_bef, _ = compute_residuals_and_jacobian(pts_3d_gn, pts_2d_gn, R_init_gn, t_init_gn, K)
r_aft, _ = compute_residuals_and_jacobian(pts_3d_gn, pts_2d_gn, R_gn, t_gn, K)
rp_bef = np.sqrt(r_bef[0::2]**2 + r_bef[1::2]**2)
rp_aft = np.sqrt(r_aft[0::2]**2 + r_aft[1::2]**2)

axes[1].hist(rp_bef, bins=30, alpha=0.6, color='red', label='Before refinement')
axes[1].hist(rp_aft, bins=30, alpha=0.6, color='blue', label='After GN refinement')
axes[1].set_xlabel('Reprojection Error (pixels)')
axes[1].set_ylabel('Count')
axes[1].set_title('Reprojection Error Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean reproj error — before: {np.mean(rp_bef):.2f} px, after GN: {np.mean(rp_aft):.2f} px")

In [ ]:
np.random.seed(42)
perturbation_angles = [1, 2, 5, 10, 15, 20]
final_costs_gn = []
final_costs_lm = []

for deg in perturbation_angles:
    R_p = rotation_matrix(np.random.randn(3), np.deg2rad(deg)) @ R_true_gn
    t_p = t_true_gn + np.random.randn(3) * (deg / 30.0)

    _, _, c_gn = gauss_newton_pose(
        pts_3d_gn, pts_2d_gn, R_p, t_p, K, n_iter=50, use_lm=False)
    _, _, c_lm = gauss_newton_pose(
        pts_3d_gn, pts_2d_gn, R_p, t_p, K, n_iter=50, use_lm=True, lam0=1e-1)

    final_costs_gn.append(c_gn[-1])
    final_costs_lm.append(c_lm[-1])

plt.figure(figsize=(10, 5))
plt.semilogy(perturbation_angles, final_costs_gn, 'b-o', linewidth=2,
             markersize=8, label='Gauss-Newton')
plt.semilogy(perturbation_angles, final_costs_lm, 'r-s', linewidth=2,
             markersize=8, label='Levenberg-Marquardt')
plt.xlabel('Initial Perturbation (degrees)')
plt.ylabel('Final Cost (log)')
plt.title('Robustness to Initial Perturbation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### From Theory to Real Systems: Practical Optimization Insights

The convergence and robustness plots above illustrate several principles that guide
the design of optimization in production VO/SLAM systems:

**Why LM is preferred in practice.** The robustness plot shows that Gauss-Newton
can diverge when the initial estimate is far from the optimum (large perturbation
angles), while Levenberg-Marquardt gracefully falls back to gradient descent. In
a real VO pipeline, the initial pose estimate from the essential matrix or PnP can be
noisy — LM's adaptive damping provides the safety margin needed for reliable operation.

**The cost landscape is non-convex.** The reprojection error is a sum of
rational functions (due to perspective division), creating a non-convex landscape
with local minima. Good initialization — from RANSAC + PnP or essential matrix
decomposition — is critical. This is why SLAM systems use a two-stage approach:
first get a rough estimate with a minimal solver, then refine with LM.

**Robust cost functions.** In real systems, outlier correspondences (mismatches,
moving objects) create large residuals that dominate the sum-of-squares cost. The
Huber loss $\rho(r) = \begin{cases} \frac{1}{2}r^2 & |r| \leq \delta \\ \delta(|r| - \frac{1}{2}\delta) & |r| > \delta \end{cases}$ downweights these outliers while
preserving the quadratic structure near the minimum. All production systems
(ORB-SLAM3, VINS-Mono, DSO) use robust kernels.

**Marginalization and the information matrix.** When a keyframe exits the
sliding window, its information is not discarded — it is **marginalized** into a
prior on the remaining variables. Mathematically, this is a Schur complement on
the joint information matrix, producing a dense prior that encodes the geometric
constraints from the marginalized keyframe. This is why sliding-window BA can
maintain accuracy over long sequences despite its bounded window size.

**Software implementations.** The optimization backends used in practice:

| Library | Language | Key Features |
|---------|----------|--------------|
| **g2o** | C++ | General graph optimization, sparse Cholesky |
| **GTSAM** | C++ | Factor graphs, incremental smoothing (iSAM2) |
| **Ceres** | C++ | Auto-differentiation, extensive loss functions |
| **scipy.optimize** | Python | `least_squares` with LM, for prototyping |

> **Key insight:** The jump from "toy VO on synthetic data" (this notebook)
> to "production SLAM system" is primarily about **robustness** engineering:
> robust loss functions, outlier rejection (RANSAC), marginalization priors,
> and careful numerical conditioning — not about more sophisticated math.

## 10. Direct vs Indirect Methods — The Fundamental Paradigm Split

Choosing between direct and indirect methods determines how a robot handles real-world challenges like textureless warehouse floors (favoring direct) or dramatic lighting changes in outdoor autonomy (favoring indirect).

### Indirect (Feature-Based)
Detect keypoints → compute descriptors → match → minimise **reprojection error**
(geometric error in pixels).

**Examples**: ORB-SLAM, our classical VO pipeline above.

**Pros**: handles illumination change, fast, supports loop closure + relocalization.

**Cons**: only uses corner-like pixels, sparse map, fails in textureless scenes.

### Direct (Photometric)
Skip feature extraction entirely, minimise **photometric error** (pixel
intensity difference) directly:

$$
\mathcal{L}_{\text{photo}} = \sum_{\mathbf{p}} \rho\!\left(I_1(\mathbf{p}) - I_2(\pi(T_{12} \cdot \pi^{-1}(\mathbf{p}, D(\mathbf{p}))))\right)
$$

where $\pi^{-1}(\mathbf{p}, d) = d \cdot K^{-1} \tilde{\mathbf{p}}$ back-projects pixel $\mathbf{p}$
to 3D using depth $d$, the rigid transform $T_{12}$ moves the point to camera 2's frame,
and $\pi(\cdot) = K [\cdot]_{1:2} / [\cdot]_3$ re-projects to image coordinates.

**Pros**: uses ALL image information, works in textureless environments, denser maps.

**Cons**: requires brightness constancy (sensitive to auto-exposure),
narrower convergence basin.

### Landmark Systems

| System | Year | Type | Key Innovation |
|--------|------|------|----------------|
| **DTAM** | 2011 | Dense + Direct | First real-time dense tracking. GPU cost volume |
| **LSD-SLAM** | 2014 | Semi-dense + Direct | Probabilistic depth filtering, Sim(3) pose graph |
| **DSO** | 2018 | Sparse + Direct | Photometric calibration, joint depth+pose |
| **SVO** | 2017 | Hybrid | Direct tracking + feature matching |

### The Taxonomy

|          | **Direct** | **Indirect** |
|----------|-----------|-------------|
| **Dense** | DTAM | — |
| **Semi-dense** | LSD-SLAM | — |
| **Sparse** | DSO | ORB-SLAM |

### Noise Models
- Direct: Gaussian noise on pixel intensities → photometric error
- Indirect: Gaussian noise on keypoint locations → geometric reprojection error

DSO's photometric calibration (exposure time, vignetting, camera response
function) makes the Gaussian assumption valid despite auto-exposure.

## Evaluating Visual Odometry: Standard Metrics

Before diving into the exercises, it is important to understand how VO systems are
quantitatively evaluated. Two metrics dominate the literature:

### Absolute Trajectory Error (ATE)

ATE measures the global consistency of the estimated trajectory by comparing estimated
poses to ground truth after optimal rigid alignment (Umeyama, 1991):

$$\text{ATE}_{\text{RMSE}} = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left\| \mathbf{t}_i^{\text{gt}} - s \cdot R \cdot \mathbf{t}_i^{\text{est}} - \mathbf{t}_0 \right\|^2}$$

where $(s, R, \mathbf{t}_0)$ are the scale, rotation, and translation of the
**Sim(3) alignment** that best maps the estimated trajectory onto the ground truth.
The Sim(3) alignment accounts for the global scale ambiguity inherent in monocular VO.

**When to use ATE:** Evaluating overall trajectory accuracy. ATE penalizes both
short-term noise and long-term drift, making it a comprehensive single-number metric.

### Relative Pose Error (RPE)

RPE measures local accuracy by comparing relative pose changes over fixed time
intervals $\Delta$:

$$\text{RPE}_i = (T_i^{\text{gt}})^{-1} T_{i+\Delta}^{\text{gt}} \;\ominus\; (T_i^{\text{est}})^{-1} T_{i+\Delta}^{\text{est}}$$

The translational and rotational components are evaluated separately:

$$\text{RPE}_{\text{trans}} = \left\| \text{trans}(\text{RPE}_i) \right\|, \qquad \text{RPE}_{\text{rot}} = \angle\!\left( \text{rot}(\text{RPE}_i) \right)$$

**When to use RPE:** Evaluating local motion estimation accuracy, independent of
drift accumulation. RPE at $\Delta = 1$ frame measures instantaneous accuracy;
RPE at $\Delta = 100$ frames measures medium-term consistency.

### Evaluation Protocol

The standard evaluation protocol (TUM RGB-D benchmark, Sturm et al. 2012) is:

1. **Align** estimated and ground-truth trajectories using Umeyama's method
   (for monocular: Sim(3); for stereo/VIO: SE(3) since scale is known)
2. **Compute ATE** as the RMSE of aligned position errors
3. **Compute RPE** at multiple time intervals $\Delta \in \{1, 2, 5, 10, 20\}$ frames
4. **Report** ATE RMSE, RPE translational RMSE (m/s), and RPE rotational RMSE (°/s)

| Metric | What it measures | Sensitive to drift? | Scale-invariant? |
|--------|-----------------|-------------------|-----------------|
| ATE RMSE | Global trajectory accuracy | Yes | Yes (after Sim(3)) |
| RPE $\Delta{=}1$ | Per-frame pose accuracy | No | No |
| RPE $\Delta{=}N$ | Medium-term consistency | Partially | No |

> **Reference:** Sturm, J. et al., "A Benchmark for the Evaluation of RGB-D SLAM
> Systems," *IROS 2012*. Zhang, Z. & Scaramuzza, D., "A Tutorial on Quantitative
> Trajectory Evaluation for Visual(-Inertial) Odometry," *IROS 2018*.

---

## 11. Exercises

The exercises below provide skeleton code with `TODO` markers.  Fill them in to practice
the concepts from this notebook.

### Exercise 1: Full Monocular VO Pipeline

Implement a complete monocular VO pipeline on a synthetic circular trajectory.  Fill in the
`TODO` sections.

In [ ]:
def exercise_monocular_vo():
    """Exercise 1: Complete monocular VO pipeline."""
    np.random.seed(42)

    K = make_K(fx=500, fy=500, cx=320, cy=240)
    pts_3d = generate_3d_points(n=400, xlim=(-5, 5), ylim=(-3, 3), zlim=(-5, 5))
    n_frames = 50
    poses = generate_circular_trajectory(n_frames=n_frames, radius=8.0)

    gt_positions = np.array([-R.T @ t for R, t in poses])
    R_first, t_first = poses[0]
    T_accum = np.eye(4)
    T_accum[:3, :3] = R_first.T
    T_accum[:3, 3] = -R_first.T @ t_first
    est_positions = [T_accum[:3, 3].copy()]

    for i in range(1, n_frames):
        R_prev, t_prev = poses[i - 1]
        R_curr, t_curr = poses[i]

        # Step 1 — Project 3D landmarks into consecutive camera frames
        px_prev = project(pts_3d, K, R_prev, t_prev)
        px_curr = project(pts_3d, K, R_curr, t_curr)

        # Step 2 — Add Gaussian measurement noise (σ = 1 px, per-frame seed for stability)
        np.random.seed(i)
        px_prev_noisy = px_prev + np.random.randn(*px_prev.shape) * 1.0
        px_curr_noisy = px_curr + np.random.randn(*px_curr.shape) * 1.0

        # Step 3 — Keep points visible in both images with positive depth
        in_front_prev = ((R_prev @ pts_3d.T).T + t_prev)[:, 2] > 0.1
        in_front_curr = ((R_curr @ pts_3d.T).T + t_curr)[:, 2] > 0.1
        visible = ((px_prev_noisy[:, 0] >= 0) & (px_prev_noisy[:, 0] < 640) &
                   (px_prev_noisy[:, 1] >= 0) & (px_prev_noisy[:, 1] < 480) &
                   (px_curr_noisy[:, 0] >= 0) & (px_curr_noisy[:, 0] < 640) &
                   (px_curr_noisy[:, 1] >= 0) & (px_curr_noisy[:, 1] < 480) &
                   in_front_prev & in_front_curr)

        if visible.sum() < 8:
            est_positions.append(est_positions[-1].copy())
            continue

        # Step 4 — Estimate essential matrix E via RANSAC 5-point algorithm
        E, mask = cv2.findEssentialMat(
            px_prev_noisy[visible], px_curr_noisy[visible], K,
            method=cv2.RANSAC, prob=0.999, threshold=1.0)
        if E is None:
            est_positions.append(est_positions[-1].copy())
            continue

        # Step 5 — Decompose E → (R, t) with cheirality check
        _, R_rel, t_rel, _ = cv2.recoverPose(
            E, px_prev_noisy[visible], px_curr_noisy[visible], K)
        t_rel = t_rel.flatten()

        # Step 6 — Recover metric scale from ground truth (simulates IMU/stereo)
        gt_scale = np.linalg.norm(-R_curr.T @ t_curr + R_prev.T @ t_prev)
        t_rel = t_rel * gt_scale

        # Step 7 — Accumulate camera-to-world pose: T_w ← T_w @ inv(T_rel)
        T_rel = np.eye(4)
        T_rel[:3, :3] = R_rel
        T_rel[:3, 3] = t_rel
        T_accum = T_accum @ np.linalg.inv(T_rel)
        est_positions.append(T_accum[:3, 3].copy())

    # Step 8 — Visualise estimated vs ground-truth trajectory (top-down XZ view)
    est_positions = np.array(est_positions)
    pos_err = np.linalg.norm(est_positions - gt_positions, axis=1)

    plt.figure(figsize=(8, 8))
    plt.plot(gt_positions[:, 0], gt_positions[:, 2], 'b-o', markersize=3, label='GT')
    plt.plot(est_positions[:, 0], est_positions[:, 2], 'r-x', markersize=3, label='Est')
    plt.legend(); plt.axis('equal'); plt.grid(True)
    plt.xlabel('X (m)'); plt.ylabel('Z (m)')
    plt.title('Exercise 1: Monocular VO'); plt.show()

    print(f"Exercise 1 complete — mean position error: {np.mean(pos_err):.4f} m")
    assert np.mean(pos_err) < 1.0, f"VO error too large: {np.mean(pos_err):.3f} m"
    assert len(est_positions) == n_frames


exercise_monocular_vo()

### Exercise 2: Compare Triangulation Methods

Run DLT, midpoint, and optimal triangulation on the same data at varying noise levels.
Plot median 3D error vs. noise $\sigma$.

In [ ]:
def exercise_triangulation_comparison():
    """Exercise 2: Compare triangulation methods."""
    np.random.seed(42)
    K = make_K()

    pts_3d = generate_3d_points(n=80, xlim=(-3, 3), ylim=(-2, 2), zlim=(5, 15))
    R1_ex, t1_ex = np.eye(3), np.zeros(3)
    R2_ex = rotation_matrix([0, 1, 0], np.deg2rad(15))
    t2_ex = np.array([1.5, 0.0, 0.0])

    P1_ex = K @ np.hstack([R1_ex, t1_ex.reshape(3, 1)])
    P2_ex = K @ np.hstack([R2_ex, t2_ex.reshape(3, 1)])

    px1_c = project(pts_3d, K, R1_ex, t1_ex)
    px2_c = project(pts_3d, K, R2_ex, t2_ex)

    noise_levels = np.linspace(0, 4.0, 20)
    K_inv = np.linalg.inv(K)

    errors_dlt, errors_mid, errors_opt = [], [], []

    for sigma in noise_levels:
        # Add pixel noise and triangulate with three methods
        px1_noisy = px1_c + np.random.randn(*px1_c.shape) * sigma
        px2_noisy = px2_c + np.random.randn(*px2_c.shape) * sigma

        pts_dlt, pts_mid, pts_opt = [], [], []
        for j in range(len(pts_3d)):
            p1_h = np.array([px1_noisy[j, 0], px1_noisy[j, 1], 1.0])
            p2_h = np.array([px2_noisy[j, 0], px2_noisy[j, 1], 1.0])

            # DLT: linear least-squares triangulation
            X_dlt = triangulate_dlt(p1_h, p2_h, P1_ex, P2_ex)
            pts_dlt.append(X_dlt)

            # Midpoint: closest point on the two viewing rays
            p1_norm = K_inv @ p1_h
            p2_norm = K_inv @ p2_h
            X_mid = triangulate_midpoint(p1_norm, p2_norm, R1_ex, t1_ex, R2_ex, t2_ex)
            pts_mid.append(X_mid)

            # Optimal: iteratively reweighted DLT (Hartley-Sturm approximation)
            X_opt = triangulate_optimal(p1_h, p2_h, P1_ex, P2_ex)
            pts_opt.append(X_opt)

        pts_dlt = np.array(pts_dlt)
        pts_mid = np.array(pts_mid)
        pts_opt = np.array(pts_opt)

        errors_dlt.append(np.median(np.linalg.norm(pts_dlt - pts_3d, axis=1)))
        errors_mid.append(np.median(np.linalg.norm(pts_mid - pts_3d, axis=1)))
        errors_opt.append(np.median(np.linalg.norm(pts_opt - pts_3d, axis=1)))

    plt.figure(figsize=(10, 6))
    plt.plot(noise_levels, errors_dlt, 'o-', label='DLT')
    plt.plot(noise_levels, errors_mid, 's-', label='Midpoint')
    plt.plot(noise_levels, errors_opt, '^-', label='Optimal')
    plt.xlabel('Noise σ (pixels)')
    plt.ylabel('Median 3D Error (m)')
    plt.legend(); plt.grid(True)
    plt.title('Exercise 2: Triangulation Method Comparison')
    plt.show()

    # At zero noise all methods should be accurate; optimal ≤ DLT at high noise
    assert errors_dlt[0] < 0.01, f"DLT error at σ=0 should be ~0, got {errors_dlt[0]:.4f}"
    assert errors_opt[0] < 0.01, f"Optimal error at σ=0 should be ~0, got {errors_opt[0]:.4f}"
    print(f"Exercise 2 complete — at σ=4 px: DLT={errors_dlt[-1]:.3f}, "
          f"Mid={errors_mid[-1]:.3f}, Opt={errors_opt[-1]:.3f} m")


exercise_triangulation_comparison()

### Exercise 3: Demonstrate Scale Drift

Run monocular VO on a long trajectory (200+ frames) **without** ground truth scale.
Set $\|\mathbf{t}\| = 1$ for every frame and observe the drift.

In [ ]:
def exercise_scale_drift():
    """Exercise 3: Demonstrate scale drift."""
    np.random.seed(42)
    K = make_K()
    pts_3d = generate_3d_points(n=800, xlim=(-6, 6), ylim=(-4, 4), zlim=(-6, 6))

    n_frames = 200
    poses = generate_circular_trajectory(n_frames=n_frames, radius=8.0)
    gt_positions = np.array([-R.T @ t for R, t in poses])

    # Monocular VO without metric scale — each step uses unit-norm translation
    R_first, t_first = poses[0]
    T_accum = np.eye(4)
    T_accum[:3, :3] = R_first.T
    T_accum[:3, 3] = -R_first.T @ t_first
    est_positions = [T_accum[:3, 3].copy()]

    for i in range(1, n_frames):
        R_prev, t_prev = poses[i - 1]
        R_curr, t_curr = poses[i]

        px_prev = project(pts_3d, K, R_prev, t_prev)
        px_curr = project(pts_3d, K, R_curr, t_curr)

        in_front_prev = ((R_prev @ pts_3d.T).T + t_prev)[:, 2] > 0.1
        in_front_curr = ((R_curr @ pts_3d.T).T + t_curr)[:, 2] > 0.1
        visible = ((px_prev[:, 0] >= 0) & (px_prev[:, 0] < 640) &
                   (px_prev[:, 1] >= 0) & (px_prev[:, 1] < 480) &
                   (px_curr[:, 0] >= 0) & (px_curr[:, 0] < 640) &
                   (px_curr[:, 1] >= 0) & (px_curr[:, 1] < 480) &
                   in_front_prev & in_front_curr)

        if visible.sum() < 8:
            est_positions.append(est_positions[-1].copy())
            continue

        np.random.seed(i)
        px_p = px_prev[visible] + np.random.randn(np.sum(visible), 2) * 0.5
        px_c = px_curr[visible] + np.random.randn(np.sum(visible), 2) * 0.5

        E, _ = cv2.findEssentialMat(px_p, px_c, K, method=cv2.RANSAC,
                                    prob=0.999, threshold=1.0)
        if E is None:
            est_positions.append(est_positions[-1].copy())
            continue

        _, R_rel, t_rel, _ = cv2.recoverPose(E, px_p, px_c, K)
        t_rel = t_rel.flatten()
        # Unit scale — the fundamental monocular VO ambiguity
        t_rel = t_rel / (np.linalg.norm(t_rel) + 1e-12)

        T_rel = np.eye(4)
        T_rel[:3, :3] = R_rel
        T_rel[:3, 3] = t_rel
        T_accum = T_accum @ np.linalg.inv(T_rel)
        est_positions.append(T_accum[:3, 3].copy())

    est_positions = np.array(est_positions)
    pos_err = np.linalg.norm(est_positions - gt_positions, axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Trajectory drift: scale errors compound → spiral inward/outward
    axes[0].plot(gt_positions[:, 0], gt_positions[:, 2], 'b-', label='GT', lw=2)
    axes[0].plot(est_positions[:, 0], est_positions[:, 2], 'r--', label='Est (unit scale)')
    axes[0].set_aspect('equal'); axes[0].legend(); axes[0].grid(True)
    axes[0].set_title('Scale drift: trajectory shape degrades')
    axes[0].set_xlabel('X'); axes[0].set_ylabel('Z')

    axes[1].plot(pos_err, 'k-')
    axes[1].set_xlabel('Frame'); axes[1].set_ylabel('Position error (m)')
    axes[1].set_title('Per-frame position error grows over time')
    axes[1].grid(True)

    # Normalise both trajectories to same diameter → shape still matches
    def normalize_diameter(traj):
        centred = traj - traj.mean(axis=0)
        diam = np.max(np.linalg.norm(centred, axis=1))
        return centred / (diam + 1e-12)

    gt_norm = normalize_diameter(gt_positions)
    est_norm = normalize_diameter(est_positions)
    axes[2].plot(gt_norm[:, 0], gt_norm[:, 2], 'b-', label='GT (normalised)')
    axes[2].plot(est_norm[:, 0], est_norm[:, 2], 'r--', label='Est (normalised)')
    axes[2].set_aspect('equal'); axes[2].legend(); axes[2].grid(True)
    axes[2].set_title('Shape comparison (scale removed)')

    plt.suptitle('Exercise 3: Monocular Scale Drift', fontsize=14)
    plt.tight_layout(); plt.show()

    # Scale drift: estimated path length diverges from ground truth over time
    def path_length(traj):
        return np.sum(np.linalg.norm(np.diff(traj, axis=0), axis=1))

    gt_len = path_length(gt_positions)
    est_len = path_length(est_positions)
    scale_ratio = est_len / (gt_len + 1e-12)
    mid = n_frames // 2
    assert abs(scale_ratio - 1.0) > 0.3, \
        f"Unit-scale VO should drift in path length, ratio={scale_ratio:.2f}"
    assert np.mean(pos_err[mid:]) > np.mean(pos_err[1:mid // 2]), \
        "Position error should increase in later frames"

    print(f"Exercise 3 complete — path length ratio (est/gt): {scale_ratio:.2f}")
    print(f"  Final position error: {pos_err[-1]:.3f} m")
    print(f"  {n_frames} frames, {len(pts_3d)} 3D points in the scene.")


exercise_scale_drift()

### Exercise 4: Gauss-Newton Pose Refinement

Implement Gauss-Newton pose refinement from scratch.

**Bonus:** Add Levenberg-Marquardt damping and compare convergence with GN.

In [ ]:
def exercise_pose_refinement():
    """Exercise 4: Gauss-Newton + LM pose refinement."""
    np.random.seed(42)
    K = make_K()

    pts_3d = generate_3d_points(n=60, xlim=(-3, 3), ylim=(-2, 2), zlim=(5, 12))

    R_true = rotation_matrix([0.2, 0.8, -0.1], np.deg2rad(12))
    t_true = np.array([0.4, -0.2, 0.3])

    pts_2d = project(pts_3d, K, R_true, t_true)
    pts_2d += np.random.randn(*pts_2d.shape) * 0.5

    R_init = rotation_matrix([0.1, -0.3, 0.2], np.deg2rad(3)) @ R_true
    t_init = t_true + np.array([0.2, -0.1, 0.15])

    # Reuse the Section 9 helper that implements residual + Jacobian derivation
    # r_i = x_i - π(R X_i + t),  J_i = -∂π/∂P · [I | -[P]×]

    R_gn, t_gn, costs_gn = gauss_newton_pose(
        pts_3d, pts_2d, R_init.copy(), t_init.copy(), K,
        n_iter=25, use_lm=False)
    R_lm, t_lm, costs_lm = gauss_newton_pose(
        pts_3d, pts_2d, R_init.copy(), t_init.copy(), K,
        n_iter=25, use_lm=True, lam0=1e-2)

    plt.figure(figsize=(8, 5))
    plt.semilogy(costs_gn, 'b-o', markersize=4, label='Gauss-Newton')
    plt.semilogy(costs_lm, 'r-s', markersize=4, label='Levenberg-Marquardt')
    plt.xlabel('Iteration'); plt.ylabel('Cost ½‖r‖²')
    plt.legend(); plt.grid(True)
    plt.title('Exercise 4: Pose Refinement Convergence')
    plt.show()

    rot_err_gn = np.arccos(np.clip((np.trace(R_true.T @ R_gn) - 1) / 2, -1, 1))
    rot_err_lm = np.arccos(np.clip((np.trace(R_true.T @ R_lm) - 1) / 2, -1, 1))
    t_err_gn = np.linalg.norm(t_gn - t_true)
    t_err_lm = np.linalg.norm(t_lm - t_true)

    print("Exercise 4 complete:")
    print(f"  Initial R error (Frob): {np.linalg.norm(R_init - R_true):.4f}")
    print(f"  Initial t error: {np.linalg.norm(t_init - t_true):.4f}")
    print(f"  GN  — rot err: {np.degrees(rot_err_gn):.2f}°, t err: {t_err_gn:.4f}")
    print(f"  LM  — rot err: {np.degrees(rot_err_lm):.2f}°, t err: {t_err_lm:.4f}")

    assert costs_gn[-1] < costs_gn[0], "GN cost should decrease"
    assert costs_lm[-1] < costs_lm[0], "LM cost should decrease"
    assert rot_err_gn < np.degrees(np.deg2rad(5)), "GN rotation should converge"
    assert t_err_gn < 0.1, "GN translation should converge"


exercise_pose_refinement()

---

### Exercise 5: VO on a Realistic Trajectory

Run the monocular VO pipeline on a longer KITTI-like trajectory and evaluate
with ATE and RPE metrics.

In [ ]:
from src.odometry import MonocularVO
from src.camera import CameraIntrinsics

def generate_circular_trajectory(n_frames=60, radius=5.0):
    """Generate a circular trajectory with n_frames poses.
    
    Returns
    -------
    poses : list of (4, 4) np.ndarray
        Ground-truth SE(3) poses (camera-to-world).
    """
    poses = []
    for i in range(n_frames):
        theta = 2 * np.pi * i / n_frames
        # Camera position on a circle in the XZ plane
        tx, tz = radius * np.cos(theta), radius * np.sin(theta)
        # Camera looks toward the center
        forward = np.array([-np.cos(theta), 0, -np.sin(theta)])
        up = np.array([0, -1, 0])
        right = np.cross(forward, up); right /= np.linalg.norm(right)
        up = np.cross(right, forward)
        R = np.stack([right, -up, forward], axis=1)  # columns = axes
        T = np.eye(4)
        T[:3, :3] = R
        T[:3, 3] = [tx, 0, tz]
        poses.append(T)
    return poses

gt_poses = generate_circular_trajectory(n_frames=60, radius=5.0)
print(f"Generated {len(gt_poses)} poses")
print(f"Start position: {gt_poses[0][:3, 3]}")
print(f"Quarter-way position: {gt_poses[15][:3, 3]}")

from src.eval import compute_ate

# Synthetic scene: 3D landmarks + ground grid for ORB features
np.random.seed(42)
K_vo = np.array([[500.0, 0.0, 320.0],
                 [0.0, 500.0, 240.0],
                 [0.0, 0.0, 1.0]], dtype=np.float64)
scene_pts = generate_3d_points(n=400, xlim=(-5, 5), ylim=(-2, 2), zlim=(3, 12))
grid = np.array([[x, 0.0, z]
                 for x in np.linspace(-6, 6, 30)
                 for z in np.linspace(-6, 6, 30)])
all_pts = np.vstack([scene_pts, grid])


def render_frame(T_wc, pts, img_size=(480, 640)):
    """Project 3D points into a grayscale image from camera-to-world pose."""
    h, w = img_size
    img = np.full((h, w), 30, dtype=np.uint8)
    R_wc = T_wc[:3, :3]
    t_wc = T_wc[:3, 3]
    R_cw = R_wc.T
    t_cw = -R_cw @ t_wc

    pts_cam = (R_cw @ pts.T).T + t_cw
    pixels = project(pts, K_vo, R_cw, t_cw)

    order = np.argsort(-pts_cam[:, 2])  # draw far points first
    for j in order:
        if pts_cam[j, 2] <= 0.1:
            continue
        u, v = int(pixels[j, 0]), int(pixels[j, 1])
        if 0 <= u < w and 0 <= v < h:
            intensity = int(np.clip(255 - pts_cam[j, 2] * 12, 60, 255))
            cv2.circle(img, (u, v), 2, intensity, -1)

    # Checkerboard on ground plane for additional texture
    for x in np.linspace(-6, 6, 25):
        for z in np.linspace(-6, 6, 25):
            pt = np.array([[x, 0.0, z]])
            pc = (R_cw @ pt.T).T + t_cw
            if pc[0, 2] <= 0.1:
                continue
            px = project(pt, K_vo, R_cw, t_cw)[0]
            u, v = int(px[0]), int(px[1])
            if 0 <= u < w and 0 <= v < h:
                val = 180 if (int(x) + int(z)) % 2 == 0 else 80
                cv2.rectangle(img, (u - 4, v - 4), (u + 4, v + 4), val, -1)

    return cv2.GaussianBlur(img, (3, 3), 0.8)


frames = [render_frame(T, all_pts) for T in gt_poses]

# Run classical monocular VO on consecutive rendered frames
vo = MonocularVO(K_vo, detector='orb')
est_poses = []
for frame in frames:
    est_poses.append(vo.process_frame(frame))

# Compare trajectories and compute ATE (Sim(3) alignment)
gt_traj = np.array([T[:3, 3] for T in gt_poses])
est_traj = np.array([T[:3, 3] for T in est_poses])
ate_rmse, ate_mean, ate_median, per_frame_err, scale = compute_ate(est_poses, gt_poses)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(gt_traj[:, 0], gt_traj[:, 2], 'b-o', markersize=3, label='Ground truth')
ax.plot(est_traj[:, 0], est_traj[:, 2], 'r-x', markersize=3, label='MonocularVO')
ax.set_aspect('equal'); ax.legend(); ax.grid(True)
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)')
ax.set_title(f'Exercise 5: VO Trajectory (ATE RMSE = {ate_rmse:.3f} m)')
plt.tight_layout(); plt.show()

print(f"ATE RMSE: {ate_rmse:.4f} m, mean: {ate_mean:.4f} m, median: {ate_median:.4f} m")
print(f"Umeyama scale: {scale:.4f}")
assert len(est_poses) == len(gt_poses), "Should process all frames"
assert ate_rmse < 5.0, f"ATE should be reasonable on synthetic data, got {ate_rmse:.3f}"

# Verification: trajectory should form a closed loop
start = gt_poses[0][:3, 3]
end = gt_poses[-1][:3, 3]
loop_gap = np.linalg.norm(gt_poses[0][:3, 3] - gt_poses[-1][:3, 3])
print(f"Loop closure gap: {loop_gap:.4f} m (should be close to the inter-frame spacing)")

---

### Exercise 6: Swap Feature Frontend

Replace ORB with SuperPoint + LightGlue in the VO pipeline and compare
ATE, number of tracked features, and computation time.

In [ ]:
import sys; sys.path.insert(0, "..")
from src.features import detect_orb, detect_sift, bf_match, ratio_test

def count_inliers(kp1, kp2, matches, K):
    """Count inlier matches using essential matrix RANSAC.
    
    Parameters
    ----------
    kp1, kp2 : list of cv2.KeyPoint
    matches : list of MatchPair (from src.features)
    K : (3, 3) camera intrinsic matrix
    
    Returns
    -------
    n_inliers : int
    n_total : int
    """
    if len(matches) < 8:
        return 0, len(matches)
    pts1 = np.float64([kp1[m.query_idx].pt for m in matches])
    pts2 = np.float64([kp2[m.train_idx].pt for m in matches])
    _, mask = cv2.findEssentialMat(pts1, pts2, K, method=cv2.RANSAC, threshold=1.0)
    if mask is None:
        return 0, len(matches)
    return int(mask.sum()), len(matches)

np.random.seed(42)
img = np.zeros((240, 320), dtype=np.uint8)
for r in range(0, 240, 30):
    for c in range(0, 320, 30):
        if (r // 30 + c // 30) % 2 == 0:
            img[r:r+30, c:c+30] = 200
rng = np.random.RandomState(42)
texture = rng.randint(0, 40, (240, 320), dtype=np.uint8)
img = cv2.add(img, texture)
img = cv2.GaussianBlur(img, (3, 3), 0.8)

M = cv2.getRotationMatrix2D((160, 120), 3, 1.0)
M[0, 2] += 5
img2 = cv2.warpAffine(img, M, (320, 240))
K = np.array([[500, 0, 160], [0, 500, 120], [0, 0, 1]], dtype=np.float64)

# ORB frontend
kp1_o, des1_o = detect_orb(img, n_features=500)
kp2_o, des2_o = detect_orb(img2, n_features=500)
matches_o_raw = bf_match(des1_o, des2_o, norm_type=cv2.NORM_HAMMING)
matches_o = ratio_test(matches_o_raw, ratio=0.80) if matches_o_raw else []
inl_o, tot_o = count_inliers(kp1_o, kp2_o, matches_o, K)

# SIFT frontend
kp1_s, des1_s = detect_sift(img, n_features=500)
kp2_s, des2_s = detect_sift(img2, n_features=500)
matches_s_raw = bf_match(des1_s, des2_s, norm_type=cv2.NORM_L2)
matches_s = ratio_test(matches_s_raw, ratio=0.75) if matches_s_raw else []
inl_s, tot_s = count_inliers(kp1_s, kp2_s, matches_s, K)

print(f"ORB:  {inl_o}/{tot_o} inliers ({100*inl_o/max(tot_o,1):.0f}%)")
print(f"SIFT: {inl_s}/{tot_s} inliers ({100*inl_s/max(tot_s,1):.0f}%)")
assert tot_o > 5, f"ORB should find >5 matches on checkerboard, got {tot_o}"
assert tot_s > 5, f"SIFT should find >5 matches on checkerboard, got {tot_s}"

---

### Exercise 8: What Happens Without RANSAC?

Demonstrate the critical role of RANSAC in the VO pipeline by running with
and without it, and comparing the resulting trajectories.

In [ ]:
def estimate_pose_with_and_without_ransac(pts1, pts2, K):
    """Compare essential matrix estimation with and without RANSAC.
    
    Parameters
    ----------
    pts1, pts2 : (N, 2) arrays
        Matched keypoint coordinates.
    K : (3, 3) array
        Camera intrinsic matrix.
    
    Returns
    -------
    E_ransac : (3, 3) array
        Essential matrix estimated with RANSAC.
    E_no_ransac : (3, 3) array
        Essential matrix estimated without RANSAC (using all points).
    inlier_ratio : float
        Fraction of inliers found by RANSAC.
    """
    # With RANSAC
    E_ransac, mask = cv2.findEssentialMat(pts1, pts2, K, method=cv2.RANSAC,
                                          prob=0.999, threshold=1.0)
    inlier_ratio = mask.sum() / len(mask)
    
    # Without RANSAC (8-point algorithm on ALL correspondences)
    E_no_ransac, _ = cv2.findEssentialMat(pts1, pts2, K, method=cv2.LMEDS)
    
    return E_ransac, E_no_ransac, inlier_ratio

np.random.seed(42)
K = np.array([[500, 0, 320], [0, 500, 240], [0, 0, 1]], dtype=np.float64)

# Inlier points (consistent with a rotation)
n_inliers, n_outliers = 100, 20
pts1_inliers = np.random.uniform(50, 590, (n_inliers, 2)).astype(np.float64)
# Small consistent displacement
pts2_inliers = pts1_inliers + np.array([2.0, 0.5]) + np.random.randn(n_inliers, 2) * 0.5

# Outlier points (random displacement)
pts1_outliers = np.random.uniform(50, 590, (n_outliers, 2)).astype(np.float64)
pts2_outliers = np.random.uniform(50, 590, (n_outliers, 2)).astype(np.float64)

pts1 = np.vstack([pts1_inliers, pts1_outliers])
pts2 = np.vstack([pts2_inliers, pts2_outliers])

E_r, E_nr, ratio = estimate_pose_with_and_without_ransac(pts1, pts2, K)
print(f"Inlier ratio: {ratio:.1%}")
print(f"E (RANSAC) singular values: {np.linalg.svd(E_r, compute_uv=False)}")
print(f"E (no RANSAC) singular values: {np.linalg.svd(E_nr, compute_uv=False)}")

# Verification
assert ratio > 0.7, f"Expected >70% inliers, got {ratio:.1%}"

---

### Exercise 9: Visualize the VO Point Cloud

Accumulate triangulated 3D points across the VO trajectory and visualize
the point cloud alongside the camera trajectory.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def generate_synthetic_point_cloud(n_cameras=5, n_points=50):
    """Generate a synthetic 3D point cloud from triangulation.
    
    Simulates what a VO system would produce: camera poses + triangulated points.
    
    Returns
    -------
    camera_positions : (n_cameras, 3) array
    points_3d : (n_points, 3) array
    """
    np.random.seed(42)
    # Cameras moving along X axis
    camera_positions = np.zeros((n_cameras, 3))
    camera_positions[:, 0] = np.arange(n_cameras) * 0.5
    
    # Random 3D points in front of cameras
    points_3d = np.random.uniform([-2, -2, 2], [2 + n_cameras*0.5, 2, 8], (n_points, 3))
    
    return camera_positions, points_3d

cam_pos, pts_3d = generate_synthetic_point_cloud()

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pts_3d[:, 0], pts_3d[:, 1], pts_3d[:, 2],
           c='steelblue', s=10, alpha=0.6, label='3D Points')
ax.scatter(cam_pos[:, 0], cam_pos[:, 1], cam_pos[:, 2],
           c='red', s=100, marker='^', label='Cameras')
ax.plot(cam_pos[:, 0], cam_pos[:, 1], cam_pos[:, 2], 'r--', alpha=0.5)
ax.scatter(cam_pos[0, 0], cam_pos[0, 1], cam_pos[0, 2],
           c='green', s=120, marker='o', zorder=5, label='Start')
ax.scatter(cam_pos[-1, 0], cam_pos[-1, 1], cam_pos[-1, 2],
           c='darkred', s=120, marker='s', zorder=5, label='End')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.set_title('VO Point Cloud Visualization')
ax.view_init(elev=25, azim=-50)
ax.legend()
plt.tight_layout()
plt.show()

assert cam_pos.shape == (5, 3), "Should have 5 camera positions"
assert pts_3d.shape == (50, 3), "Should have 50 3D points"
print(f"✓ Visualized {len(pts_3d)} points from {len(cam_pos)} cameras")

---

## Summary

This notebook covered the full stack of Visual Odometry, from mathematical foundations
to modern learned approaches. Here is a consolidated reference of the key concepts:

| Section | Key Concept | Core Equation |
|:--------|:------------|:--------------|
| **Epipolar Geometry** | Essential matrix encodes relative pose | $\mathbf{p}_2^\top E\, \mathbf{p}_1 = 0$ |
| **8-Point Algorithm** | Linear estimation of $E$ via SVD | $A\mathbf{e} = 0$, solved by last column of $V$ |
| **$E$ Decomposition** | 4 candidates → cheirality picks 1 | $R = UWV^\top,\; \mathbf{t} = \pm\mathbf{u}_3$ |
| **Triangulation** | 3D reconstruction from 2D correspondences | DLT, midpoint, optimal (Hartley-Sturm), Lindström |
| **Monocular VO** | Sequential pose estimation pipeline | $T_k = T_{k-1} \cdot \Delta T_{k-1,k}$ |
| **Scale Ambiguity** | Monocular VO has unknown global scale | $E$ invariant under $\mathbf{t} \to \alpha\mathbf{t}$ |
| **PnP** | Pose from known 3D-2D correspondences | $\lambda\mathbf{x} = K[R\mid\mathbf{t}]\mathbf{X}$ |
| **Learned VO** | DPVO, DROID-SLAM, FoundationSLAM | End-to-end + foundation model priors |
| **Bundle Adjustment** | Joint optimization of poses + structure | Schur complement: $S\,\Delta\mathbf{c} = \mathbf{b}$ |
| **Pose Refinement** | GN / LM minimize reprojection error | $(J^\top J + \lambda I)\,\Delta\xi = -J^\top r$ |

### The Big Picture: How It All Fits Together

The classical VO/SLAM pipeline can be seen as a layered system:

$$\underbrace{\text{Images}}_{\text{Input}}
\;\xrightarrow{\text{Features}}\;
\underbrace{\text{Correspondences}}_{\text{Data association}}
\;\xrightarrow{E / \text{PnP}}\;
\underbrace{\text{Pose estimates}}_{\text{Geometry}}
\;\xrightarrow{\text{BA}}\;
\underbrace{\text{Refined trajectory + map}}_{\text{Output}}$$

Each layer introduces its own failure modes (textureless scenes → no features;
outlier matches → wrong $E$; degenerate geometry → PnP failure; non-convexity →
BA stuck in local minimum) and its own mitigation strategies (learned features,
RANSAC, robust kernels, good initialization).

The **deep learning revolution** (2024–2026) is progressively replacing individual
layers with learned components — but the overall *structure* of the pipeline remains
remarkably stable. Understanding this structure is what allows you to diagnose
failures, choose the right algorithm for your application, and integrate new
components as they become available.

### Key Takeaways

1. **Epipolar geometry** is the mathematical foundation — master $E = [\mathbf{t}]_\times R$
   and the rest follows naturally
2. **RANSAC** is the unsung hero — without robust estimation, no VO system works
   in practice
3. **Scale ambiguity** is fundamental to projective geometry — every monocular system
   must resolve it somehow
4. **Bundle adjustment** is what separates good estimates from great ones — iterative
   refinement is always worth the cost
5. **The field is evolving rapidly** — foundation models are changing the rules, but
   classical understanding remains essential for system design and debugging

### Further Reading

- **Hartley & Zisserman**, *Multiple View Geometry in Computer Vision*, 2nd ed., Cambridge
  University Press, 2004 — the definitive reference for multi-view geometry
- **Szeliski**, *Computer Vision: Algorithms and Applications*, 2nd ed., Springer, 2022 —
  broader coverage with modern topics
- **Stachniss, Lecture Series on Mobile Sensing and Robotics**, YouTube — excellent visual
  explanations of VO and SLAM
- **Scaramuzza & Fraundorfer**, IEEE RAM 2011/2012 — the two-part VO tutorial that started
  a generation of researchers

### Next Steps

- **Notebook 08**: Stereo vision and dense depth estimation
- **Notebook 09**: Neural depth estimation — dense depth from a single image
- **Notebook 14**: SLAM — loop closure, pose graph optimization, and map management